# Black Women & Graves’ Disease — CRISP-DM Notebook (Instructional Edition)

> **Synthetic-data note (read first):** This notebook generates and analyzes **synthetic** patient-like records to demonstrate a complete CRISP-DM workflow. It is **not clinical evidence**, not a prevalence estimate, and not suitable for medical decision-making.

## What you will learn
- How to structure a health-disparities analysis using **CRISP-DM** (Business → Data → Prep → Modeling → Evaluation → Deployment).
- How to generate and validate a **synthetic dataset** that encodes hypothesized disparity patterns for instructional purposes.
- How to evaluate a simple predictive model and interpret results *as a demonstration*, with explicit limitations.

## Who this is for
- Learners practicing an end-to-end data analysis process (EDA → modeling → evaluation → reporting).
- Reviewers who want a readable, scannable walkthrough with clear phase boundaries.

## Prerequisites
- Python basics (variables, functions, DataFrames)
- Libraries used: `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`

## How to use this notebook
1. Run the **Reproducibility & Environment** cell once.
2. Execute top-to-bottom for a deterministic narrative (some randomness is controlled via seeds).
3. Read the **Checkpoint Summaries** at the end of major phases.

## Epistemic status
- **Data:** Synthetic (generated in-notebook).
- **Claims:** Instructional demonstrations of workflow and interpretation patterns.
- **Limitations:** Patterns are *encoded assumptions*; validate with real datasets and domain review for real-world use.


---

## Design Principles: Narrative-First Analytics

This notebook is written to be *read*, not just executed. I treat **accessibility as cognitive design**: the structure of the notebook is part of the method, because reasoning under complexity depends on how information is staged, explained, and verified.

Each phase uses an explicit **Analytical Objective** and a **Rationale & Payoff** to externalize intent, assumptions, and tradeoffs. Where feasible, the analysis compares multiple modeling approaches as **complementary cognitive lenses** (stability, controlled innovation, boundary-probing exploration) so disagreement and uncertainty become visible rather than implicit.



In [ ]:
# Reproducibility & Environment
import sys, platform, random
import numpy as np
import pandas as pd
import matplotlib
import seaborn as sns
import sklearn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)
print("scikit-learn:", sklearn.__version__)
print("Seed:", SEED)


## Table of contents
- [Part 1 — CRISP-DM walkthrough](#part-1--crisp-dm-walkthrough-conceptual--setup)
  - [Business understanding](#business-understanding)
  - [Data understanding](#data-understanding)
  - [Data preparation](#data-preparation)
  - [Modeling](#modeling)
  - [Evaluation](#evaluation)
  - [Deployment](#deployment-conceptual)
  - [Checkpoint summary](#checkpoint-summary)
- [Part 2 — Synthetic data generation (self-instruct loop)](#part-2--synthetic-data-generation-self-instruct-loop)
- [Part 3 — Analysis using synthetic data](#part-3--analysis-using-synthetic-data)
- [Appendix — Demo script outline](#appendix--demo-script-outline-madmall-context)
- [Appendix — Gemini chat (provenance)](#appendix--gemini-chat-provenance)


# Part 1 — CRISP-DM Walkthrough (Conceptual + Setup)

## Business understanding

#### Analytical Objective:
Be clear about what this analysis is for and what it is not. Define who this serves, which disparity questions we are exploring (diagnosis timing, treatment differences, outcomes), and where the epistemic boundary sits since the data is synthetic.

> **Rationale & Payoff:**  
> If we don't define the boundary now, we risk confusing demonstration with evidence. This step keeps us honest about what we are simulating, what we are testing, and what claims we are not allowed to make. It protects the integrity of the entire notebook.

## Data understanding

#### Analytical Objective:
Define exactly what the dataset must contain for a disparity analysis to make sense. Identify required fields, subgroup coverage, expected patterns, and validity constraints, while explicitly acknowledging that this is synthetic stand-in data.


> **Rationale & Payoff:**  
> Data that is merely “available” is not the same as data that is fit for purpose. This step ensures the structure supports the questions we care about. It prevents us from building models on a foundation that cannot actually answer the disparity questions.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a random seed for reproducibility
np.random.seed(42)

# Define the number of records
n_records = 1000

# Simulate demographic data
races = ['White', 'Black', 'Asian', 'Hispanic', 'Other']
genders = ['Female', 'Male']
race_distribution = [0.6, 0.2, 0.1, 0.05, 0.05] # Adjust distribution as needed
gender_distribution = [0.7, 0.3] # Graves' is more common in females

data = {
    'PatientID': range(1, n_records + 1),
    'Age_at_Diagnosis': np.random.normal(loc=45, scale=15, size=n_records).astype(int).clip(5, 90),
    'Race': np.random.choice(races, size=n_records, p=race_distribution),
    'Gender': np.random.choice(genders, size=n_records, p=gender_distribution)
}

df = pd.DataFrame(data)

# Introduce a higher proportion of females, especially Black females, as Graves' is more prevalent
female_patients_indices = df[df['Gender'] == 'Female'].index
black_female_patients_indices = df[(df['Gender'] == 'Female') & (df['Race'] == 'Black')].index

# Simulate Graves' Disease diagnosis - making it more likely in females and potentially varying by race
df['Graves_Diagnosis'] = 0 # Default to no diagnosis

# Increase diagnosis likelihood for females
df.loc[female_patients_indices, 'Graves_Diagnosis'] = np.random.choice([0, 1], size=len(female_patients_indices), p=[0.6, 0.4])

# Further increase diagnosis likelihood for Black females (simulating a potential disparity or focus)
df.loc[black_female_patients_indices, 'Graves_Diagnosis'] = np.random.choice([0, 1], size=len(black_female_patients_indices), p=[0.4, 0.6])


# Simulate treatment information (simplified)
treatments = ['Antithyroid Drugs', 'Radioactive Iodine', 'Surgery', 'None']
df['Treatment'] = np.random.choice(treatments, size=n_records, p=[0.4, 0.3, 0.1, 0.2])

# Adjust treatment distribution for diagnosed patients
diagnosed_indices = df[df['Graves_Diagnosis'] == 1].index
df.loc[diagnosed_indices, 'Treatment'] = np.random.choice(treatments[:-1], size=len(diagnosed_indices), p=[0.5, 0.3, 0.2]) # Only treatment for diagnosed

# Simulate outcomes (simplified: Remission, Persistent Hyperthyroidism, Hypothyroidism)
outcomes = ['Remission', 'Persistent Hyperthyroidism', 'Hypothyroidism']
df['Outcome'] = 'No Diagnosis' # Default outcome

# Assign outcomes to diagnosed patients (simulating potential disparities)
remission_probs = {'White': 0.6, 'Black': 0.4, 'Asian': 0.5, 'Hispanic': 0.55, 'Other': 0.5}
persistent_probs = {'White': 0.2, 'Black': 0.4, 'Asian': 0.25, 'Hispanic': 0.2, 'Other': 0.25}
hypothyroid_probs = {'White': 0.2, 'Black': 0.2, 'Asian': 0.25, 'Hispanic': 0.25, 'Other': 0.25}


for index in diagnosed_indices:
    race = df.loc[index, 'Race']
    outcome_probs = [remission_probs[race], persistent_probs[race], hypothyroid_probs[race]]
    df.loc[index, 'Outcome'] = np.random.choice(outcomes, p=outcome_probs)

# Simulate socioeconomic factors (simplified: Income Level)
income_levels = ['Low', 'Medium', 'High']
df['Income_Level'] = np.random.choice(income_levels, size=n_records, p=[0.3, 0.5, 0.2])

# Introduce a potential bias: Lower income may correlate with worse outcomes, especially for Black women
low_income_black_female_indices = df[(df['Income_Level'] == 'Low') & (df['Race'] == 'Black') & (df['Gender'] == 'Female') & (df['Graves_Diagnosis'] == 1)].index

# Increase likelihood of less favorable outcomes for low-income Black females with Graves'
for index in low_income_black_female_indices:
     df.loc[index, 'Outcome'] = np.random.choice(['Persistent Hyperthyroidism', 'Hypothyroidism'], p=[0.6, 0.4]) # Higher chance of worse outcomes


# Display initial information about the simulated data
display(df.head())
display(df.info())
display(df.describe(include='all'))

# Initial exploration of key columns
print("\nValue counts for categorical columns:")
for col in ['Race', 'Gender', 'Graves_Diagnosis', 'Treatment', 'Outcome', 'Income_Level']:
    if col in df.columns:
        print(f"\n{col}:")
        display(df[col].value_counts())

# Check for missing values
print("\nMissing values per column:")
display(df.isnull().sum())

# Initial Visualizations
# Distribution of Graves' Diagnosis by Gender and Race
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Race', hue='Graves_Diagnosis', palette='viridis')
plt.title('Graves\' Diagnosis by Race and Gender')
plt.xlabel('Race')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Graves Diagnosis', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Gender', hue='Graves_Diagnosis', palette='viridis')
plt.title('Graves\' Diagnosis by Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.legend(title='Graves Diagnosis', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

# Distribution of Outcomes for Diagnosed Patients by Race
diagnosed_df = df[df['Graves_Diagnosis'] == 1].copy()
if not diagnosed_df.empty:
    plt.figure(figsize=(12, 7))
    sns.countplot(data=diagnosed_df, x='Race', hue='Outcome', palette='viridis')
    plt.title('Outcomes for Diagnosed Patients by Race')
    plt.xlabel('Race')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Outcome')
    plt.tight_layout()
    plt.show()
else:
    print("\nNo diagnosed patients in the simulated data to plot outcomes.")

# Age distribution by Diagnosis Status
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='Age_at_Diagnosis', hue='Graves_Diagnosis', multiple='stack', kde=True, palette='viridis')
plt.title('Age Distribution by Graves\' Diagnosis Status')
plt.xlabel('Age at Diagnosis')
plt.ylabel('Count')
plt.legend(title='Graves Diagnosis', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

## Data preparation

#### Analytical Objective:
Create a clean, analysis-ready table with consistent data types, explicit encoding decisions, and clearly derived features (including a justified subgroup flag), while preserving lineage from simulated fields to model inputs.


> **Rationale & Payoff:**  
> Preparation is where hidden assumptions creep in. Making choices explicit reduces leakage, prevents ambiguous encodings, and creates a stable “contract” that EDA and modeling can rely on without rework.


In [ ]:
# Re-create the DataFrame from the original data dictionary to ensure original columns are present
df = pd.DataFrame(data)

# Simulate Graves' Disease diagnosis and outcomes again to ensure consistency with previous steps
# Introduce a higher proportion of females, especially Black females, as Graves' is more prevalent
female_patients_indices = df[df['Gender'] == 'Female'].index
black_female_patients_indices = df[(df['Gender'] == 'Female') & (df['Race'] == 'Black')].index

# Simulate Graves' Disease diagnosis - making it more likely in females and potentially varying by race
df['Graves_Diagnosis'] = 0 # Default to no diagnosis

# Increase diagnosis likelihood for females
df.loc[female_patients_indices, 'Graves_Diagnosis'] = np.random.choice([0, 1], size=len(female_patients_indices), p=[0.6, 0.4])

# Further increase diagnosis likelihood for Black females (simulating a potential disparity or focus)
df.loc[black_female_patients_indices, 'Graves_Diagnosis'] = np.random.choice([0, 1], size=len(black_female_patients_indices), p=[0.4, 0.6])

# Simulate treatment information (simplified)
treatments = ['Antithyroid Drugs', 'Radioactive Iodine', 'Surgery', 'None']
df['Treatment'] = np.random.choice(treatments, size=n_records, p=[0.4, 0.3, 0.1, 0.2])

# Adjust treatment distribution for diagnosed patients
diagnosed_indices = df[df['Graves_Diagnosis'] == 1].index
df.loc[diagnosed_indices, 'Treatment'] = np.random.choice(treatments[:-1], size=len(diagnosed_indices), p=[0.5, 0.3, 0.2]) # Only treatment for diagnosed

# Simulate outcomes (simplified: Remission, Persistent Hyperthyroidism, Hypothyroidism)
outcomes = ['Remission', 'Persistent Hyperthyroidism', 'Hypothyroidism']
df['Outcome'] = 'No Diagnosis' # Default outcome

# Assign outcomes to diagnosed patients (simulating potential disparities)
# Using the previously defined probability dictionaries
for index in diagnosed_indices:
    race = df.loc[index, 'Race']
    outcome_probs = [remission_probs[race], persistent_probs[race], hypothyroid_probs[race]]
    df.loc[index, 'Outcome'] = np.random.choice(outcomes, p=outcome_probs)

# Simulate socioeconomic factors (simplified: Income Level)
income_levels = ['Low', 'Medium', 'High']
df['Income_Level'] = np.random.choice(income_levels, size=n_records, p=[0.3, 0.5, 0.2])

# Introduce a potential bias: Lower income may correlate with worse outcomes, especially for Black women
low_income_black_female_indices = df[(df['Income_Level'] == 'Low') & (df['Race'] == 'Black') & (df['Gender'] == 'Female') & (df['Graves_Diagnosis'] == 1)].index

# Increase likelihood of less favorable outcomes for low-income Black females with Graves'
for index in low_income_black_female_indices:
     df.loc[index, 'Outcome'] = np.random.choice(['Persistent Hyperthyroidism', 'Hypothyroidism'], p=[0.6, 0.4]) # Higher chance of worse outcomes


# 2. Handle any missing values.
# Check for missing values
print("Missing values before handling:")
display(df.isnull().sum())
# As noted previously, there are no missing values in this simulated dataset, so no action is needed here.

# 3. Address potential outliers in numerical columns, specifically 'Age_at_Diagnosis'.
# Analyze the distribution of 'Age_at_Diagnosis'
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['Age_at_Diagnosis'])
plt.title('Boxplot of Age at Diagnosis')
plt.xlabel('Age')
plt.show()

# Given the nature of age, values between 5 and 90 are reasonable for diagnosis.
# The simulation already clipped the values to this range. We will not remove or transform based on outliers here,
# as the current range is clinically plausible and extreme ages at diagnosis could be relevant.

# 4. Encode categorical variables ('Race', 'Gender', 'Treatment', 'Outcome', 'Income_Level')
# Use one-hot encoding for nominal categorical variables: 'Race', 'Gender', 'Treatment', 'Income_Level'
# Keep all dummy columns for 'Gender' to create 'Is_Black_Female' feature
df = pd.get_dummies(df, columns=['Race', 'Gender', 'Treatment', 'Income_Level'], drop_first=False)

# For 'Outcome', which has a hierarchical structure (No Diagnosis vs outcomes for Diagnosed),
# and where 'No Diagnosis' is essentially a non-event for the analysis of outcomes in diagnosed patients,
# we can handle this by filtering or using a different encoding if needed for specific models.
# For now, we will one-hot encode it as well, as it's a multi-class outcome for the diagnosed group.
df = pd.get_dummies(df, columns=['Outcome'], prefix='Outcome')

# Handle the 'Outcome_No Diagnosis' column created by one-hot encoding.
# This column indicates patients who were not diagnosed with Graves' Disease.
# For analysis focused on outcomes *of diagnosed patients*, this column can be used to filter,
# or it can be kept as a feature if predicting diagnosis status is part of a broader model.
# We will keep it for now, as it implicitly captures the 'Graves_Diagnosis' information.
# We can drop the original 'Graves_Diagnosis' column as 'Outcome_No Diagnosis' serves a similar purpose
# in distinguishing diagnosed vs. non-diagnosed patients, and the other Outcome columns only apply to diagnosed.
if 'Graves_Diagnosis' in df.columns:
    df = df.drop('Graves_Diagnosis', axis=1)


# 5. Create new features if deemed beneficial for the analysis.
# Create a specific flag for "Black Female"
# Use the one-hot encoded columns created in the previous step
df['Is_Black_Female'] = ((df['Race_Black'] == 1) & (df['Gender_Female'] == 1)).astype(int)

# 6. Ensure the data types are appropriate for each column.
# Check current data types
print("\nData types before ensuring:")
display(df.dtypes)
# The one-hot encoding process typically results in appropriate data types (int or uint8).
# 'Age_at_Diagnosis' is already int. No specific action needed here based on current dtypes.

# 7. Display the first few rows and the information of the preprocessed DataFrame.
print("\nPreprocessed DataFrame head:")
display(df.head())

print("\nPreprocessed DataFrame info:")
display(df.info())

## Exploratory data analysis (eda) & feature engineering

#### Analytical Objective:
Examine how the key variables actually behave. Look at differences in age at diagnosis, treatment patterns, outcome distributions, and income associations to decide which features are having the most impact and thus justified for modeling.

> **Rationale & Payoff:**
>
> EDA keeps us from inventing or claiming patterns that aren’t there. We add features only when the data clearly shows they influence the outcome—not just because a model is capable of using them. This step forces us to ground our decisions in evidence and keeps the analysis focused on what the data actually supports.



In [ ]:
# Create a DataFrame for diagnosed patients using the one-hot encoded column
diagnosed_df = df[df['Outcome_No Diagnosis'] == 0].copy()

# Ensure 'Is_Black_Female' is in the diagnosed_df
# This column was already created during preprocessing, but ensure it's in the subset
if 'Is_Black_Female' not in diagnosed_df.columns:
     diagnosed_df['Is_Black_Female'] = ((diagnosed_df['Race_Black'] == 1) & (diagnosed_df['Gender_Female'] == 1)).astype(int)


# 1. Visualize the distribution of 'Age_at_Diagnosis' for Black women with Graves' Disease versus other groups with Graves' Disease.
plt.figure(figsize=(10, 6))
sns.histplot(data=diagnosed_df, x='Age_at_Diagnosis', hue='Is_Black_Female', multiple='dodge', kde=True, palette='viridis', common_norm=False)
plt.title('Age at Diagnosis Distribution: Black Female vs. Other Diagnosed Patients')
plt.xlabel('Age at Diagnosis')
plt.ylabel('Density')
plt.legend(title='Group', labels=['Other Diagnosed', 'Black Female Diagnosed'])
plt.tight_layout()
plt.show()

# 2. Analyze the distribution of 'Treatment' types for diagnosed Black women compared to other diagnosed patients.
# Melt the DataFrame to have a single column for treatment types
treatment_cols = [col for col in diagnosed_df.columns if col.startswith('Treatment_')]
treatment_melted = diagnosed_df.melt(id_vars=['PatientID', 'Is_Black_Female'], value_vars=treatment_cols, var_name='Treatment_Type', value_name='Is_Treated')
treatment_melted = treatment_melted[treatment_melted['Is_Treated'] == 1]
treatment_melted['Treatment_Type'] = treatment_melted['Treatment_Type'].str.replace('Treatment_', '')


plt.figure(figsize=(12, 7))
sns.countplot(data=treatment_melted, x='Treatment_Type', hue='Is_Black_Female', palette='viridis')
plt.title('Treatment Distribution: Black Female vs. Other Diagnosed Patients')
plt.xlabel('Treatment Type')
plt.ylabel('Count')
plt.legend(title='Group', labels=['Other Diagnosed', 'Black Female Diagnosed'])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 3. Visualize the distribution of 'Outcome' categories for diagnosed Black women compared to other diagnosed patients.
# Melt the DataFrame to have a single column for outcome types
outcome_cols = [col for col in diagnosed_df.columns if col.startswith('Outcome_') and col != 'Outcome_No Diagnosis']
outcome_melted = diagnosed_df.melt(id_vars=['PatientID', 'Is_Black_Female'], value_vars=outcome_cols, var_name='Outcome_Type', value_name='Is_Outcome')
outcome_melted = outcome_melted[outcome_melted['Is_Outcome'] == 1]
outcome_melted['Outcome_Type'] = outcome_melted['Outcome_Type'].str.replace('Outcome_', '')


plt.figure(figsize=(12, 7))
sns.countplot(data=outcome_melted, x='Outcome_Type', hue='Is_Black_Female', palette='viridis')
plt.title('Outcome Distribution: Black Female vs. Other Diagnosed Patients')
plt.xlabel('Outcome Type')
plt.ylabel('Count')
plt.legend(title='Group', labels=['Other Diagnosed', 'Black Female Diagnosed'])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 4. Explore the relationship between 'Income_Level' and 'Outcome' specifically for Black women with Graves' Disease.
black_female_diagnosed_df = diagnosed_df[diagnosed_df['Is_Black_Female'] == 1].copy()

if not black_female_diagnosed_df.empty:
    # Melt for outcome
    black_female_outcome_melted = black_female_diagnosed_df.melt(id_vars=['PatientID'] + [col for col in black_female_diagnosed_df.columns if col.startswith('Income_')],
                                                                value_vars=outcome_cols, var_name='Outcome_Type', value_name='Is_Outcome')
    black_female_outcome_melted = black_female_outcome_melted[black_female_outcome_melted['Is_Outcome'] == 1]
    black_female_outcome_melted['Outcome_Type'] = black_female_outcome_melted['Outcome_Type'].str.replace('Outcome_', '')

    # Determine income level for each patient
    income_cols = [col for col in black_female_diagnosed_df.columns if col.startswith('Income_')]
    def get_income_level(row):
        for col in income_cols:
            if row[col] == 1:
                return col.replace('Income_', '')
        return 'Unknown' # Should not happen with one-hot encoding

    black_female_outcome_melted['Income_Level'] = black_female_outcome_melted.apply(get_income_level, axis=1)

    # Order income levels for plotting
    income_order = ['Low', 'Medium', 'High']
    black_female_outcome_melted['Income_Level'] = pd.Categorical(black_female_outcome_melted['Income_Level'], categories=income_order, ordered=True)


    plt.figure(figsize=(10, 6))
    sns.countplot(data=black_female_outcome_melted, x='Income_Level', hue='Outcome_Type', palette='viridis')
    plt.title('Outcome Distribution by Income Level for Black Female Diagnosed Patients')
    plt.xlabel('Income Level')
    plt.ylabel('Count')
    plt.legend(title='Outcome Type')
    plt.tight_layout()
    plt.show()
else:
    print("\nNo Black Female diagnosed patients in the data to plot Income vs. Outcome.")

# 5. Create a visualization to show the proportion of diagnosed patients who are Black women compared to other demographics.
diagnosed_counts = diagnosed_df['Is_Black_Female'].value_counts().reset_index()
diagnosed_counts.columns = ['Is_Black_Female', 'Count']
diagnosed_counts['Group'] = diagnosed_counts['Is_Black_Female'].apply(lambda x: 'Black Female Diagnosed' if x == 1 else 'Other Diagnosed')

plt.figure(figsize=(7, 7))
plt.pie(diagnosed_counts['Count'], labels=diagnosed_counts['Group'], autopct='%1.1f%%', startangle=140, colors=sns.color_palette('viridis', 2))
plt.title('Proportion of Black Female Diagnosed Patients within the Diagnosed Group')
plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

# 6. Calculate and display descriptive statistics for 'Age_at_Diagnosis' for the Black female diagnosed group and compare them to the overall diagnosed group.
print("\nDescriptive Statistics for Age at Diagnosis:")
print("\nOverall Diagnosed Group:")
display(diagnosed_df['Age_at_Diagnosis'].describe())

print("\nBlack Female Diagnosed Group:")
if not black_female_diagnosed_df.empty:
    display(black_female_diagnosed_df['Age_at_Diagnosis'].describe())
else:
    print("No Black Female diagnosed patients in the data for descriptive statistics.")

# 7. Consider creating interaction terms between 'Is_Black_Female' and other relevant features (e.g., 'Age_at_Diagnosis', 'Income_Level')
# This step is for consideration and noting down ideas based on EDA.
# Based on the visualizations, if there appear to be different patterns for Black females
# across age, treatment, outcome, or income compared to other groups, interaction terms could be useful.
# For example, if the effect of income level on outcome appears different for Black females.
# We will note this down as a potential feature engineering step for the modeling phase.

# 8. Based on the insights from EDA, identify any other potential features to engineer or transformations to apply.
# Note down ideas based on the visualizations and descriptive statistics:
# - Interaction terms: 'Is_Black_Female' * 'Age_at_Diagnosis'
# - Interaction terms: 'Is_Black_Female' * 'Income_Level_Low', 'Is_Black_Female' * 'Income_Level_Medium', 'Is_Black_Female' * 'Income_Level_High'
# - Potentially group less frequent treatment types if needed for modeling.
# - Explore creating age groups/bins instead of using raw age, depending on model requirements.
# - If additional data were available, features related to comorbidities or duration of symptoms before diagnosis could be relevant.

print("\nPotential Feature Engineering Ideas for Modeling (based on EDA):")
print("- Interaction terms: 'Is_Black_Female' * 'Age_at_Diagnosis'")
print("- Interaction terms between 'Is_Black_Female' and Income Level dummy variables.")
print("- Grouping of less frequent treatment types.")
print("- Creating age groups/bins.")
print("- (If external data available) Features on comorbidities or symptom duration.")

## Modeling

#### Analytical Objective:
Train a small set of models aligned to the defined endpoints, using clean inputs and a reproducible protocol, so we can compare how different modeling assumptions affect what patterns become visible.


> **Rationale & Payoff:**
>
> Models turn hypotheses into something testable. Comparing them shows us how assumptions—like linearity versus nonlinearity—change what we can see. The goal is understanding, not chasing a single “best” score.


### Modeling Frame: Models as Cognitive Lenses

#### Analytical Objective
Use multiple model types intentionally, treating each as a different way of “seeing” the data rather than competitors in a leaderboard.

>**Rationale & Payoff**
>
>Different models highlight different structures. A simple model provides interpretability and stability. More flexible models test whether added complexity reveals additional signal. This framing keeps modeling aligned with learning rather than optimization for its own sake.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Define features (X) and target variable(s) (y) for the modeling task.
# We will focus on predicting the outcome (Remission or Persistent Hyperthyroidism or Hypothyroidism)
# for diagnosed patients. 'No Diagnosis' is excluded as it's not an outcome of Graves' Disease treatment.

# Filter the DataFrame to include only diagnosed patients
diagnosed_df = df[df['Outcome_No Diagnosis'] == 0].copy()

# Define the target variables (Outcomes). We will treat this as a multi-class classification problem for simplicity.
# The target will be the specific outcome category for diagnosed patients.
# We need to convert the one-hot encoded outcome columns back to a single categorical column for the target.
outcome_cols = [col for col in diagnosed_df.columns if col.startswith('Outcome_') and col != 'Outcome_No Diagnosis']

def get_outcome(row):
    for col in outcome_cols:
        if row[col] == 1:
            return col.replace('Outcome_', '')
    return None # Should not happen for diagnosed patients

diagnosed_df['Outcome'] = diagnosed_df.apply(get_outcome, axis=1)

# Drop the individual one-hot encoded outcome columns and 'Outcome_No Diagnosis' as they are now represented by the 'Outcome' column
diagnosed_df = diagnosed_df.drop(columns=outcome_cols + ['Outcome_No Diagnosis'])

# Define features (X). Exclude PatientID and the newly created 'Outcome' column.
X = diagnosed_df.drop(columns=['PatientID', 'Outcome'])

# Define the target variable (y)
y = diagnosed_df['Outcome']

# Ensure all feature columns are numerical. Drop any remaining non-numerical columns if any.
X = X.select_dtypes(include=np.number)

# 2. Split the data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("Training set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)

# 3. Select appropriate models.
# Since the target variable 'Outcome' is categorical (Remission, Persistent Hyperthyroidism, Hypothyroidism),
# classification models are appropriate.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Support Vector Machine": SVC(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# 4. Train the selected model(s) on the training data.
trained_models = {}
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"{name} trained.")

# 5. Make predictions on the testing data using the trained model(s).
predictions = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    print(f"\nPredictions made for {name}.")

In keeping with the Narrative-First
Analytics approach described at the start of this notebook, the models below are not treated as a competition to find a single “best” performer.

Instead, they are used as **complementary cognitive lenses**:

- A **traditional model** (e.g., logistic regression) serves as a stable, interpretable baseline and shared reference point.
- **Enhanced-traditional models** (e.g., tree ensembles) introduce controlled flexibility to test whether additional structure improves signal capture.
- **More experimental models** (e.g., boosted or kernel-based methods) are used exploratorily to probe boundaries and surface patterns that simpler assumptions may miss.

The goal is not premature optimization, but comparative understanding: what different modeling assumptions make visible, obscure, or distort.


## Evaluation

#### Analytical Objective:
Assess how each model actually performs, especially on outcomes tied to potential disparities. Look beyond overall accuracy to see where mistakes occur and which groups or outcomes are harder to predict.

> **Rationale & Payoff:**
>
> Evaluation stops us from fooling ourselves. A high accuracy score can hide serious weaknesses. Looking at confusion matrices, recall, and class-level performance forces us to confront where the model fails—particularly where errors could obscure disparity patterns.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Iterate through the trained models and their corresponding predictions
for name, model in trained_models.items():
    print(f"--- Evaluating {name} ---")

    # Calculate and print the classification report
    print("\nClassification Report:")
    print(classification_report(y_test, predictions[name]))

    # Calculate and print the confusion matrix
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, predictions[name]))

    # Calculate and print the accuracy score
    accuracy = accuracy_score(y_test, predictions[name])
    print(f"\nAccuracy Score: {accuracy:.4f}")

    print("-" * (len(name) + 20))

# Interpretation of results in the context of business understanding and research objectives
print("\n--- Interpretation of Model Performance ---")
print("Based on the evaluation metrics (Precision, Recall, F1-score, Support, Confusion Matrix, and Accuracy), we can interpret the performance of each model:")

# Interpret each model's performance, focusing on outcomes relevant to disparities in Black women
for name in trained_models.keys():
    print(f"\nInterpretation for {name}:")
    report = classification_report(y_test, predictions[name], output_dict=True)
    matrix = confusion_matrix(y_test, predictions[name])
    accuracy = accuracy_score(y_test, predictions[name])

    print(f"- Overall Accuracy: {accuracy:.4f}")

    # Focus on key outcomes for potential disparities
    print("- Performance on Key Outcomes (Persistent Hyperthyroidism, Hypothyroidism):")
    for outcome in ['Persistent Hyperthyroidism', 'Hypothyroidism', 'Remission']:
        if outcome in report:
            print(f"  - {outcome}:")
            print(f"    - Precision: {report[outcome]['precision']:.4f}")
            print(f"    - Recall: {report[outcome]['recall']:.4f}")
            print(f"    - F1-score: {report[outcome]['f1-score']:.4f}")
            print(f"    - Support: {report[outcome]['support']}")
        else:
             print(f"  - {outcome}: No instances in test set.")


    # Relate performance to research objectives and potential disparities
    print("- Relation to Research Objectives:")
    print(f"  - The model's ability to predict '{'Persistent Hyperthyroidism'}' and '{'Hypothyroidism'}' is particularly relevant to identifying potential disparities in outcomes for Black women, as these represent less favorable outcomes compared to Remission.")
    # Add more specific interpretation based on the matrix if needed, e.g., false positives/negatives for specific classes

print("\nOverall Summary and Relation to Disparities:")
print("Reviewing the metrics across all models:")
# Summarize which models perform best for key outcomes and overall
best_overall_model = max(predictions, key=lambda name: accuracy_score(y_test, predictions[name]))
print(f"- The model with the highest overall accuracy on the test set is: {best_overall_model}")

# You can add more detailed comparison here, e.g., which model has the best recall for Persistent Hyperthyroidism, etc.
# This requires more complex parsing of the report dictionary, but for a general interpretation,
# the above metrics provide a good basis.

print("\nImplications for Identifying Disparities:")
print("The performance of these models in predicting different outcomes can help us understand if certain models are better at identifying patterns associated with less favorable outcomes, which could be linked to disparities faced by Black women. A model that performs well in predicting Persistent Hyperthyroidism or Hypothyroidism might be more useful in flagging patients who may require closer monitoring or different treatment approaches.")
print("However, it's crucial to remember that model performance alone does not prove disparities. Further analysis, including examining model predictions specifically for the 'Is_Black_Female' group and conducting statistical tests, would be necessary to draw conclusions about disparities.")

### Comparative Interpretation: What Each Model Reveals

Rather than treating performance metrics as definitive judgments, this analysis uses them as prompts for interpretation:

- **Simpler models** clarify which features have consistent, directional influence, but may miss non-linear interactions.
- **More flexible models** can capture richer structure, but often at the cost of transparency and stability across subgroups.
- **Exploratory models** may suggest patterns worth further investigation, even when predictive performance is not decisively superior.

This comparison highlights an epistemic point: differences in model behavior reflect differences in *assumptions*, not just differences in quality. Understanding those assumptions is essential in disparity-sensitive contexts.


## Deployment (conceptual)

#### Analytical Objective:
Translate findings into responsible “use pathways” (communication, decision-support concepts, interventions) while documenting ethical constraints, monitoring needs, and what real-world validation would be required before any use.


> **Rationale & Payoff:**
>
> This step turns analysis into practice without pretending the notebook is production or clinical evidence. It teaches the professional endpoint of CRISP-DM: not “a model,” but a governed, monitored, ethically constrained application of findings.


In [ ]:
# 1. Dissemination of Findings
print("1. Dissemination of Findings:")
print("Insights from this analysis, including potential disparities in outcomes for Black women with Graves' Disease, would need to be effectively communicated to relevant stakeholders.")
print("- Healthcare Providers: Share findings through medical journals, conferences, and targeted workshops. Provide practical summaries highlighting key differences in presentation, treatment response, or risk factors observed in Black female patients.")
print("- Public Health Officials: Present data on the prevalence of less favorable outcomes in specific demographic groups to inform public health campaigns, resource allocation, and policy development aimed at reducing health disparities.")
print("- Patient Advocacy Groups: Collaborate with organizations supporting Graves' Disease patients to share findings in an accessible format. This can empower patients with information and support advocacy efforts for equitable care.")
print("- Researchers: Publish detailed findings and methodology to contribute to the scientific understanding of Graves' Disease in diverse populations and encourage further research.")
print("-" * 30)

# 2. Potential Use of Trained Models in Clinical Settings
print("2. Potential Use of Trained Models in Clinical Settings:")
print("Models demonstrating good performance, especially in identifying patients at risk of Persistent Hyperthyroidism or Hypothyroidism, could potentially be integrated into clinical workflows, with careful consideration and validation.")
print("- Risk Stratification: Integrate models into Electronic Health Record (EHR) systems to flag patients, particularly Black women, who may be at higher risk of less favorable outcomes based on their characteristics (e.g., age, income level, initial treatment response).")
print("- Decision Support: Provide clinicians with model-based risk assessments at key decision points (e.g., choosing initial treatment, determining follow-up frequency). The model could suggest that a patient has a higher likelihood of a certain outcome, prompting closer monitoring or consideration of alternative treatment strategies.")
print("- Personalized Treatment Plans: While complex, in the future, models could potentially contribute to developing more personalized treatment plans by predicting individual responses to different therapies, taking into account demographic and socioeconomic factors.")
print("-" * 30)

# 3. Informing Targeted Interventions and Programs
print("3. Informing Targeted Interventions and Programs:")
print("Findings related to potential disparities can directly inform the development of targeted interventions.")
print("- Culturally Sensitive Patient Education: Develop educational materials and programs specifically tailored to the needs and contexts of Black women, addressing potential barriers to care or understanding of the disease and treatment options.")
print("- Community Health Programs: Implement outreach programs in communities with a high proportion of Black women to increase awareness of Graves' Disease symptoms, the importance of timely diagnosis, and available resources.")
print("- Healthcare Provider Training: Provide training to healthcare professionals on recognizing and addressing potential biases in care, improving communication with diverse patient populations, and understanding the specific challenges faced by Black women with Graves' Disease.")
print("- Address Socioeconomic Factors: Advocate for policies and programs that address socioeconomic determinants of health, such as improving access to affordable healthcare, transportation, and healthy food, which can indirectly impact health outcomes.")
print("-" * 30)

# 4. Ethical Considerations and Bias Mitigation
print("4. Ethical Considerations and Bias Mitigation:")
print("Deploying models trained on potentially biased data raises significant ethical concerns.")
print("- Data Bias: Acknowledge and quantify potential biases in the training data (e.g., underrepresentation of certain groups, biased outcome reporting).")
print("- Model Bias: Evaluate model performance across different demographic subgroups (e.g., Black women vs. other groups) to ensure fairness and avoid amplifying existing disparities. Do not deploy models that perform poorly or unfairly for specific groups.")
print("- Transparency and Explainability: Aim for transparent and explainable models (e.g., using techniques like SHAP or LIME) so that clinicians and patients can understand why a particular risk assessment or prediction was made.")
print("- Regular Monitoring: Continuously monitor model performance in the real world to detect any drift or emergent biases.")
print("- Inclusive Development: Involve representatives from the target population (Black women with Graves' Disease) in the development and deployment process to ensure the model and interventions are appropriate and acceptable.")
print("-" * 30)

# 5. Plan for Ongoing Monitoring and Evaluation
print("5. Plan for Ongoing Monitoring and Evaluation:")
print("Any deployed interventions or models require continuous monitoring and evaluation to ensure effectiveness and identify unintended consequences.")
print("- Outcome Tracking: Continuously track patient outcomes (remission rates, complications, etc.) for different demographic groups after implementing interventions or using models.")
print("- Model Performance Monitoring: Regularly evaluate the performance of deployed models using new data to ensure accuracy and fairness over time. Retrain models as needed.")
print("- Feedback Mechanisms: Establish feedback loops from healthcare providers and patients to understand their experiences with the interventions or models and identify areas for improvement.")
print("- Disparity Metrics: Continue to monitor key disparity metrics (e.g., differences in outcome rates between Black women and other groups) to assess the impact of interventions.")
print("- A/B Testing (if feasible): For interventions, consider controlled studies (if ethical and practical) to rigorously evaluate their effectiveness.")
print("-" * 30)

### Checkpoint Summary — Data Analysis Key Findings

*   The analysis focused on identifying potential disparities in the diagnosis, treatment, and outcomes of Graves' Disease in Black women compared to other demographics.
*   A simulated dataset of 1000 records was created, including demographic information, Graves' Disease diagnosis status, treatment received, outcome, and income level, to facilitate the analysis due to the difficulty in acquiring specific real-world data.
*   Initial data exploration confirmed the simulated higher prevalence of Graves' Disease in females, particularly Black females, and illustrated simulated disparities in outcomes across different racial groups.
*   The data preparation involved one-hot encoding of categorical variables and the creation of a new feature, 'Is\_Black\_Female', to specifically identify this subgroup.
*   Exploratory Data Analysis (EDA) included visualizations comparing Black female diagnosed patients to other diagnosed patients regarding age at diagnosis, treatment types, and outcomes. It also explored the relationship between income level and outcome specifically for Black women with Graves' Disease.
*   Within the simulated dataset, the EDA suggested potential differences in the distribution of treatments and outcomes between Black female patients and other diagnosed groups. For instance, the simulated data showed a lower proportion of Remission and higher proportions of Persistent Hyperthyroidism and Hypothyroidism among Black female patients compared to other groups.
*   Several classification models (Logistic Regression, Support Vector Machine, Random Forest, and Gradient Boosting) were trained to predict treatment outcomes (Remission, Persistent Hyperthyroidism, Hypothyroidism) for diagnosed patients.
*   Model evaluation showed varying performance, with Logistic Regression having the highest overall accuracy (0.5682) and some ability to predict Persistent Hyperthyroidism, while none of the models effectively predicted Hypothyroidism on the test set. Performance varied significantly across outcome classes and models.

### Insights or Next Steps

*   The simulation highlighted the potential for disparities in Graves' Disease outcomes for Black women, particularly concerning less favorable outcomes like Persistent Hyperthyroidism and Hypothyroidism, and their potential correlation with socioeconomic factors like income level.
*   Future steps should involve seeking and analyzing real-world clinical data to validate the patterns observed in the simulation. If real data confirms potential disparities, further investigation into the underlying factors (e.g., access to care, treatment adherence, biological differences) is crucial.

This checkpoint reflects the core principle of this notebook’s Theory of Change and Action: **accessibility as cognitive design**. By externalizing assumptions, comparing multiple modeling lenses, and narrating tradeoffs explicitly, the analysis prioritizes interpretability and auditability over premature certainty.


# Part 2 — Synthetic Data Generation (Self-Instruct Loop)

**Instructional role of Part 2:**
Teach how synthetic data can be deliberately constructed to encode hypotheses, while maintaining epistemic discipline and auditability.

## Understand the CoT Self-Instruct Process

#### Analytical Objective:
Explain the self-instruct loop as a controlled generation protocol: a generator produces candidate records, an evaluator scores them against explicit criteria, and revisions are made until the dataset satisfies the documented data contract.


> **Rationale & Payoff:**
>
> Treating synthetic generation as an iterative, criteria-driven process (not a one-shot prompt) makes the assumptions auditable: you can explain *why* a pattern exists in the synthetic data and how it relates back to stated objectives.


## Define data requirements

#### Analytical Objective:
Specify the minimum set of variables, subgroup identifiers, and endpoints required to answer the research questions—plus constraints (ranges, missingness, plausibility) that the synthetic generator must satisfy.


> **Rationale & Payoff:**
>
> Defining requirements before generation enforces analysis-first thinking: every synthetic field exists because it is needed to evaluate a question, an endpoint, or a confounder—not because it was easy to generate.


## Data Field Definitions and Desired Patterns for Synthetic Data

#### Analytical Objective:
Define each synthetic field’s meaning, type, allowable values, and intended distributional patterns—especially where subgroup differences are deliberately encoded for instructional purposes.


> **Rationale & Payoff:**
>
> Explicit field definitions create auditability: reviewers can trace observed patterns back to design intent, and learners can distinguish “encoded hypotheses” from “discovered effects.”


Based on the research questions and objectives related to disparities in diagnosis, treatment, and outcomes for Black women with Graves' Disease, the following data fields and patterns are defined for the synthetic dataset:

**1. Key Data Fields:**

*   **PatientID:** A unique identifier for each patient.
*   **Age_at_Diagnosis:** The age of the patient when diagnosed with Graves' Disease.
*   **Race:** The patient's self-reported racial category.
*   **Gender:** The patient's self-reported gender.
*   **Graves_Diagnosis:** Indicates whether the patient was diagnosed with Graves' Disease.
*   **Treatment:** The primary treatment received for Graves' Disease (if diagnosed).
*   **Outcome:** The outcome of Graves' Disease treatment (if diagnosed).
*   **Income_Level:** A categorical representation of the patient's income level, serving as a proxy for socioeconomic status.

**2. Data Types:**

*   **PatientID:** Integer
*   **Age_at_Diagnosis:** Integer
*   **Race:** Categorical (String)
*   **Gender:** Categorical (String)
*   **Graves_Diagnosis:** Binary (Integer or Boolean)
*   **Treatment:** Categorical (String)
*   **Outcome:** Categorical (String)
*   **Income_Level:** Categorical (String)

**3. Potential Values for Categorical Fields:**

*   **Race:** ['White', 'Black', 'Asian', 'Hispanic', 'Other']
*   **Gender:** ['Female', 'Male']
*   **Graves_Diagnosis:** [0 (No), 1 (Yes)]
*   **Treatment:** ['Antithyroid Drugs', 'Radioactive Iodine', 'Surgery', 'None'] (Note: 'None' is for undiagnosed patients)
*   **Outcome:** ['Remission', 'Persistent Hyperthyroidism', 'Hypothyroidism', 'No Diagnosis'] (Note: 'No Diagnosis' is for undiagnosed patients)
*   **Income_Level:** ['Low', 'Medium', 'High']

**4. Desired Relationships and Patterns:**

The synthetic data should aim to exhibit the following patterns to simulate potential disparities and influences:

*   **Higher Prevalence in Females:** A significantly higher proportion of 'Yes' in 'Graves_Diagnosis' for patients with 'Gender' = 'Female' compared to 'Male'.
*   **Focus on Black Women:** Within the 'Female' population, a slightly higher probability of 'Graves_Diagnosis' = 'Yes' for 'Race' = 'Black' compared to other racial groups, reflecting the focus of the research question (this can be adjusted based on simulated disparity levels).
*   **Age Distribution:** A plausible distribution for 'Age_at_Diagnosis', likely centered in middle age but with a range reflecting diagnoses across the lifespan.
*   **Treatment Distribution:** For diagnosed patients ('Graves_Diagnosis' = 1), a distribution across 'Treatment' types. This distribution may vary by demographic group, including potential differences for Black women (e.g., differences in access to or utilization of certain treatments).
*   **Outcome Disparities:** For diagnosed patients, the distribution of 'Outcome' should show a lower proportion of 'Remission' and higher proportions of 'Persistent Hyperthyroidism' and 'Hypothyroidism' for Black women compared to other racial/gender groups. This simulates potential disparities in treatment effectiveness or access to optimal care.
*   **Socioeconomic Influence:** 'Income_Level' should be correlated with 'Outcome' for diagnosed patients, particularly for Black women. Specifically, 'Low' income level should be associated with a higher probability of less favorable outcomes ('Persistent Hyperthyroidism', 'Hypothyroidism') within the Black female diagnosed group compared to higher income levels or other demographic groups.
*   **Consistency:** Ensure that 'Treatment' and 'Outcome' are 'None' and 'No Diagnosis' respectively for patients with 'Graves_Diagnosis' = 0.

These definitions and patterns will guide the generation of synthetic data in the subsequent steps, allowing for exploration and analysis related to the research objectives.

## Develop initial prompts

#### Analytical Objective:
Translate the data contract into clear generation instructions that constrain the model to the desired schema, plausibility bounds, and subgroup coverage.


> **Rationale & Payoff:**
>
> Well-scoped prompts reduce iteration noise and prevent the loop from “wandering.” The prompt becomes a reusable artifact: a compact specification of what the synthetic data must represent.


In [ ]:
initial_prompt = """
You are a data generation model. Your task is to create synthetic patient records for a study on Graves' Disease, with a specific focus on simulating disparities related to Black women.

Generate data entries with the following fields and characteristics:

- **PatientID:** Unique integer (start from 1)
- **Age_at_Diagnosis:** Integer between 5 and 90
- **Race:** Categorical string: 'White', 'Black', 'Asian', 'Hispanic', 'Other'
- **Gender:** Categorical string: 'Female', 'Male'
- **Graves_Diagnosis:** Binary integer: 0 (No) or 1 (Yes)
- **Treatment:** Categorical string: 'Antithyroid Drugs', 'Radioactive Iodine', 'Surgery', 'None'. If Graves_Diagnosis is 0, Treatment must be 'None'.
- **Outcome:** Categorical string: 'Remission', 'Persistent Hyperthyroidism', 'Hypothyroidism', 'No Diagnosis'. If Graves_Diagnosis is 0, Outcome must be 'No Diagnosis'.
- **Income_Level:** Categorical string: 'Low', 'Medium', 'High'

**Simulate the following relationships and patterns:**

- Graves' Disease is more prevalent in Females than Males.
- Within the Female population, slightly higher likelihood of Graves' Diagnosis for 'Black' race.
- Age at Diagnosis should generally follow a distribution centered around middle age.
- For patients with Graves' Diagnosis (1):
    - Treatment distribution should be among 'Antithyroid Drugs', 'Radioactive Iodine', and 'Surgery'.
    - Outcome distribution should be among 'Remission', 'Persistent Hyperthyroidism', and 'Hypothyroidism'.
    - **Simulate Disparities:** For Black women (Race='Black' and Gender='Female') with Graves' Diagnosis, the likelihood of 'Persistent Hyperthyroidism' or 'Hypothyroidism' should be higher than for other racial/gender groups. The likelihood of 'Remission' should be lower.
    - **Simulate Socioeconomic Influence:** For Black women with Graves' Diagnosis, 'Low' Income_Level should be associated with a higher likelihood of 'Persistent Hyperthyroidism' or 'Hypothyroidism' compared to Medium or High income levels within this group.
- For patients without Graves' Diagnosis (0):
    - Treatment must be 'None'.
    - Outcome must be 'No Diagnosis'.

**Generate data in a structured format, like a list of dictionaries or a similar clear format.**

**Example Entries (DO NOT simply copy these, generate new data following the patterns):**

```json
[
  {{
    "PatientID": 1,
    "Age_at_Diagnosis": 48,
    "Race": "White",
    "Gender": "Female",
    "Graves_Diagnosis": 1,
    "Treatment": "Radioactive Iodine",
    "Outcome": "Remission",
    "Income_Level": "Medium"
  }},
  {{
    "PatientID": 2,
    "Age_at_Diagnosis": 62,
    "Race": "Black",
    "Gender": "Female",
    "Graves_Diagnosis": 1,
    "Treatment": "Antithyroid Drugs",
    "Outcome": "Persistent Hyperthyroidism",
    "Income_Level": "Low"
  }},
  {{
    "PatientID": 3,
    "Age_at_Diagnosis": 35,
    "Race": "White",
    "Gender": "Male",
    "Graves_Diagnosis": 0,
    "Treatment": "None",
    "Outcome": "No Diagnosis",
    "Income_Level": "High"
  }},
    {{
    "PatientID": 4,
    "Age_at_Diagnosis": 55,
    "Race": "Black",
    "Gender": "Female",
    "Graves_Diagnosis": 1,
    "Treatment": "Surgery",
    "Outcome": "Hypothyroidism",
    "Income_Level": "Low"
  }}
]
```

**Generate 100 new patient records following these instructions.**
"""

print(initial_prompt)

## Implement the self-instruct loop

#### Analytical Objective:
Run an iterative generate → score → revise cycle that improves synthetic data quality against explicit acceptance criteria (schema validity, plausibility, and intended pattern coverage).


> **Rationale & Payoff:**
>
> The loop teaches responsible iteration: improvements are driven by stated criteria, not intuition. This models how to make synthetic data generation reproducible, reviewable, and aligned to analytical intent.


In [ ]:
import json

# Function to generate data using the LLM (simulated)
def generate_data(prompt, num_records):
    """
    Simulates data generation by an LLM based on a prompt.
    In a real scenario, this would involve an API call to the LLM.
    For this simulation, we'll use the previous simulation logic to generate data
    that generally follows the prompt's requirements.
    """
    np.random.seed() # Use a different seed for each generation attempt

    races = ['White', 'Black', 'Asian', 'Hispanic', 'Other']
    genders = ['Female', 'Male']
    # Adjust distributions slightly to reflect potential changes from prompt refinements
    race_distribution = [0.58, 0.22, 0.1, 0.06, 0.04] # Slightly increased Black representation
    gender_distribution = [0.75, 0.25] # Increased female representation

    data = {
        'PatientID': range(1, num_records + 1),
        'Age_at_Diagnosis': np.random.normal(loc=48, scale=12, size=num_records).astype(int).clip(10, 85), # Adjusted age distribution
        'Race': np.random.choice(races, size=num_records, p=race_distribution),
        'Gender': np.random.choice(genders, size=num_records, p=gender_distribution)
    }

    df_gen = pd.DataFrame(data)

    female_patients_indices = df_gen[df_gen['Gender'] == 'Female'].index
    black_female_patients_indices = df_gen[(df_gen['Gender'] == 'Female') & (df_gen['Race'] == 'Black')].index

    df_gen['Graves_Diagnosis'] = 0

    # Increased diagnosis likelihood for females
    df_gen.loc[female_patients_indices, 'Graves_Diagnosis'] = np.random.choice([0, 1], size=len(female_patients_indices), p=[0.5, 0.5]) # Increased likelihood

    # Further increase diagnosis likelihood for Black females
    df_gen.loc[black_female_patients_indices, 'Graves_Diagnosis'] = np.random.choice([0, 1], size=len(black_female_patients_indices), p=[0.3, 0.7]) # Further increased likelihood


    treatments = ['Antithyroid Drugs', 'Radioactive Iodine', 'Surgery', 'None']
    df_gen['Treatment'] = 'None' # Default treatment
    diagnosed_indices_gen = df_gen[df_gen['Graves_Diagnosis'] == 1].index
    df_gen.loc[diagnosed_indices_gen, 'Treatment'] = np.random.choice(treatments[:-1], size=len(diagnosed_indices_gen), p=[0.45, 0.35, 0.20]) # Adjusted treatment distribution


    outcomes = ['Remission', 'Persistent Hyperthyroidism', 'Hypothyroidism', 'No Diagnosis']
    df_gen['Outcome'] = 'No Diagnosis' # Default outcome

    # Assign outcomes to diagnosed patients (simulating potential disparities)
    # Adjusted probabilities to exaggerate disparities for simulation
    remission_probs_gen = {'White': 0.6, 'Black': 0.3, 'Asian': 0.5, 'Hispanic': 0.55, 'Other': 0.5}
    persistent_probs_gen = {'White': 0.2, 'Black': 0.4, 'Asian': 0.25, 'Hispanic': 0.2, 'Other': 0.25}
    hypothyroid_probs_gen = {'White': 0.2, 'Black': 0.3, 'Asian': 0.25, 'Hispanic': 0.25, 'Other': 0.25}


    for index in diagnosed_indices_gen:
        race = df_gen.loc[index, 'Race']
        # Ensure probabilities sum to 1 for the three outcomes for diagnosed patients
        outcome_probs_gen = [remission_probs_gen.get(race, 1/3), persistent_probs_gen.get(race, 1/3), hypothyroid_probs_gen.get(race, 1/3)]
        # Normalize probabilities just in case
        prob_sum = sum(outcome_probs_gen)
        outcome_probs_gen = [p / prob_sum for p in outcome_probs_gen]

        df_gen.loc[index, 'Outcome'] = np.random.choice(outcomes[:-1], p=outcome_probs_gen)

    income_levels = ['Low', 'Medium', 'High']
    df_gen['Income_Level'] = np.random.choice(income_levels, size=num_records, p=[0.35, 0.45, 0.20]) # Adjusted income distribution

    # Introduce a potential bias: Lower income may correlate with worse outcomes, especially for Black women
    low_income_black_female_indices_gen = df_gen[(df_gen['Income_Level'] == 'Low') & (df_gen['Race'] == 'Black') & (df_gen['Gender'] == 'Female') & (df_gen['Graves_Diagnosis'] == 1)].index

    # Further increase likelihood of less favorable outcomes for low-income Black females with Graves'
    for index in low_income_black_female_indices_gen:
         df_gen.loc[index, 'Outcome'] = np.random.choice(['Persistent Hyperthyroidism', 'Hypothyroidism'], p=[0.7, 0.3]) # Increased chance of worse outcomes

    # Convert to a list of dictionaries format to simulate LLM output structure
    generated_data_list = df_gen.to_dict('records')

    # Simulate LLM output format (e.g., JSON string)
    return json.dumps(generated_data_list, indent=2)

# Function to evaluate generated data
def evaluate_data(generated_data_string, requirements):
    """
    Evaluates the quality and adherence of generated data to specified requirements.
    Returns a dictionary of evaluation results and feedback for prompt refinement.
    """
    evaluation_results = {}
    feedback = []

    try:
        # Attempt to parse the generated data string
        generated_data = json.loads(generated_data_string)
        df_eval = pd.DataFrame(generated_data)
        evaluation_results['parsing_successful'] = True

        # Check data types and presence of columns
        for field, details in requirements['fields'].items():
            if field not in df_eval.columns:
                evaluation_results[f'{field}_present'] = False
                feedback.append(f"Missing column: {field}")
                continue
            evaluation_results[f'{field}_present'] = True
            # Basic type check (can be refined)
            if details['type'] == 'Integer' and not pd.api.types.is_integer_dtype(df_eval[field]):
                 # Check if it can be converted to integer (e.g., float representation)
                 if not pd.api.types.is_numeric_dtype(df_eval[field]) or not df_eval[field].dropna().apply(lambda x: x.is_integer()).all():
                    evaluation_results[f'{field}_type_match'] = False
                    feedback.append(f"Incorrect data type for {field}. Expected Integer.")
                 else:
                     evaluation_results[f'{field}_type_match'] = True # Can be converted
            elif details['type'] == 'Categorical (String)' and not pd.api.types.is_object_dtype(df_eval[field]):
                 evaluation_results[f'{field}_type_match'] = False
                 feedback.append(f"Incorrect data type for {field}. Expected Categorical (String).")
            elif details['type'] == 'Binary (Integer or Boolean)' and not (pd.api.types.is_integer_dtype(df_eval[field]) or pd.api.types.is_bool_dtype(df_eval[field])):
                 evaluation_results[f'{field}_type_match'] = False
                 feedback.append(f"Incorrect data type for {field}. Expected Binary (Integer or Boolean).")
            else:
                 evaluation_results[f'{field}_type_match'] = True


        # Check for required values in categorical columns
        for field, details in requirements['fields'].items():
            if details['type'] in ['Categorical (String)', 'Binary (Integer or Boolean)'] and f'{field}_present' in evaluation_results and evaluation_results[f'{field}_present']:
                allowed_values = details['values']
                if not df_eval[field].isin(allowed_values).all():
                    evaluation_results[f'{field}_values_adhere'] = False
                    invalid_values = df_eval[field][~df_eval[field].isin(allowed_values)].unique()
                    feedback.append(f"Column {field} contains invalid values: {invalid_values}. Allowed: {allowed_values}")
                else:
                    evaluation_results[f'{field}_values_adhere'] = True

        # Check simulated patterns (simplified checks)
        if 'Graves_Diagnosis' in df_eval.columns and 'Gender' in df_eval.columns:
            graves_by_gender = df_eval.groupby('Gender')['Graves_Diagnosis'].mean()
            evaluation_results['female_prevalence_higher'] = graves_by_gender.get('Female', 0) > graves_by_gender.get('Male', 0)
            if not evaluation_results['female_prevalence_higher']:
                 feedback.append("Simulated pattern mismatch: Graves' prevalence not higher in Females than Males.")

        if 'Graves_Diagnosis' in df_eval.columns and 'Race' in df_eval.columns and 'Gender' in df_eval.columns:
            black_female_graves_rate = df_eval[(df_eval['Race'] == 'Black') & (df_eval['Gender'] == 'Female')]['Graves_Diagnosis'].mean()
            other_female_graves_rate = df_eval[(df_eval['Gender'] == 'Female') & (df_eval['Race'] != 'Black')]['Graves_Diagnosis'].mean()
            evaluation_results['black_female_prevalence_slightly_higher'] = black_female_graves_rate > other_female_graves_rate
            # It's 'slightly' higher in the prompt, so a simple greater than is a basic check
            if not evaluation_results['black_female_prevalence_slightly_higher']:
                 feedback.append("Simulated pattern mismatch: Graves' prevalence not slightly higher in Black Females than other Females.")


        # Check consistency for non-diagnosed patients
        non_diagnosed_df = df_eval[df_eval['Graves_Diagnosis'] == 0]
        if not non_diagnosed_df.empty:
            evaluation_results['non_diagnosed_treatment_none'] = (non_diagnosed_df['Treatment'] == 'None').all()
            evaluation_results['non_diagnosed_outcome_no_diagnosis'] = (non_diagnosed_df['Outcome'] == 'No Diagnosis').all()
            if not evaluation_results['non_diagnosed_treatment_none']:
                 feedback.append("Consistency check failed: Non-diagnosed patients should have Treatment = 'None'.")
            if not evaluation_results['non_diagnosed_outcome_no_diagnosis']:
                 feedback.append("Consistency check failed: Non-diagnosed patients should have Outcome = 'No Diagnosis'.")


        # Check simulated outcome disparities for diagnosed Black females vs. others (simplified)
        diagnosed_df_eval = df_eval[df_eval['Graves_Diagnosis'] == 1].copy()
        if not diagnosed_df_eval.empty:
            diagnosed_df_eval['Is_Black_Female'] = ((diagnosed_df_eval['Race'] == 'Black') & (diagnosed_df_eval['Gender'] == 'Female')).astype(int)
            black_female_outcomes = diagnosed_df_eval[diagnosed_df_eval['Is_Black_Female'] == 1]['Outcome'].value_counts(normalize=True)
            other_diagnosed_outcomes = diagnosed_df_eval[diagnosed_df_eval['Is_Black_Female'] == 0]['Outcome'].value_counts(normalize=True)

            evaluation_results['black_female_worse_outcomes_simulated'] = True # Assume true initially

            # Check if Persistent Hyperthyroidism or Hypothyroidism are proportionally higher for Black females
            for outcome in ['Persistent Hyperthyroidism', 'Hypothyroidism']:
                bf_rate = black_female_outcomes.get(outcome, 0)
                other_rate = other_diagnosed_outcomes.get(outcome, 0)
                if bf_rate < other_rate * 1.1: # Check if Black female rate is at least 10% higher (threshold can be adjusted)
                     evaluation_results['black_female_worse_outcomes_simulated'] = False
                     feedback.append(f"Simulated disparity mismatch: {outcome} rate not significantly higher for Black Females.")

            # Check if Remission is proportionally lower for Black females
            bf_remission_rate = black_female_outcomes.get('Remission', 0)
            other_remission_rate = other_diagnosed_outcomes.get('Remission', 0)
            if bf_remission_rate > other_remission_rate * 0.9: # Check if Black female rate is at least 10% lower (threshold can be adjusted)
                 evaluation_results['black_female_worse_outcomes_simulated'] = False
                 feedback.append(f"Simulated disparity mismatch: Remission rate not significantly lower for Black Females.")


        # Check socioeconomic influence for Black females (simplified)
        black_female_diagnosed_eval = diagnosed_df_eval[diagnosed_df_eval['Is_Black_Female'] == 1].copy()
        if not black_female_diagnosed_eval.empty:
            low_income_bf_outcomes = black_female_diagnosed_eval[black_female_diagnosed_eval['Income_Level'] == 'Low']['Outcome'].value_counts(normalize=True)
            high_income_bf_outcomes = black_female_diagnosed_eval[black_female_diagnosed_eval['Income_Level'] == 'High']['Outcome'].value_counts(normalize=True)

            evaluation_results['low_income_bf_worse_outcomes_simulated'] = True # Assume true initially

            for outcome in ['Persistent Hyperthyroidism', 'Hypothyroidism']:
                low_income_rate = low_income_bf_outcomes.get(outcome, 0)
                high_income_rate = high_income_bf_outcomes.get(outcome, 0)
                if low_income_rate < high_income_rate * 1.1: # Check if low income rate is at least 10% higher
                    evaluation_results['low_income_bf_worse_outcomes_simulated'] = False
                    feedback.append(f"Simulated socioeconomic disparity mismatch: {outcome} rate not significantly higher for Low Income Black Females.")


    except json.JSONDecodeError:
        evaluation_results['parsing_successful'] = False
        feedback.append("Generated data is not in valid JSON format.")
    except Exception as e:
        evaluation_results['evaluation_error'] = str(e)
        feedback.append(f"An error occurred during evaluation: {e}")


    return evaluation_results, feedback

# Requirements dictionary based on the defined data requirements
data_requirements = {
    'fields': {
        'PatientID': {'type': 'Integer'},
        'Age_at_Diagnosis': {'type': 'Integer', 'range': (5, 90)},
        'Race': {'type': 'Categorical (String)', 'values': ['White', 'Black', 'Asian', 'Hispanic', 'Other']},
        'Gender': {'type': 'Categorical (String)', 'values': ['Female', 'Male']},
        'Graves_Diagnosis': {'type': 'Binary (Integer or Boolean)', 'values': [0, 1]},
        'Treatment': {'type': 'Categorical (String)', 'values': ['Antithyroid Drugs', 'Radioactive Iodine', 'Surgery', 'None']},
        'Outcome': {'type': 'Categorical (String)', 'values': ['Remission', 'Persistent Hyperthyroidism', 'Hypothyroidism', 'No Diagnosis']},
        'Income_Level': {'type': 'Categorical (String)', 'values': ['Low', 'Medium', 'High']}
    },
    'patterns': [
        'Higher prevalence of Graves_Diagnosis in Females than Males.',
        'Slightly higher likelihood of Graves_Diagnosis for Black Females.',
        'Age_at_Diagnosis distribution centered around middle age.',
        'For Graves_Diagnosis=1: Treatment distribution among Antithyroid Drugs, Radioactive Iodine, Surgery.',
        'For Graves_Diagnosis=1: Outcome distribution among Remission, Persistent Hyperthyroidism, Hypothyroidism.',
        'For Graves_Diagnosis=1 and Race=Black and Gender=Female: Higher likelihood of Persistent Hyperthyroidism/Hypothyroidism, lower likelihood of Remission.',
        'For Graves_Diagnosis=1 and Race=Black and Gender=Female and Income_Level=Low: Higher likelihood of Persistent Hyperthyroidism/Hypothyroidism.',
        'For Graves_Diagnosis=0: Treatment is None and Outcome is No Diagnosis.'
    ]
}

# Self-Instruct Loop
num_iterations = 5
num_records_per_iteration = 200 # Generate a smaller batch in each iteration
generated_data_storage = []
quality_history = []

# Initial prompt (using the one defined in the previous step)
current_prompt = initial_prompt # Assuming initial_prompt variable exists from previous step

for i in range(num_iterations):
    print(f"\n--- Iteration {i+1} ---")
    print("Generating data...")
    generated_data_string = generate_data(current_prompt, num_records_per_iteration)

    print("Evaluating data...")
    evaluation_results, feedback = evaluate_data(generated_data_string, data_requirements)

    print("Evaluation Results:")
    display(evaluation_results)
    print("\nFeedback for Refinement:")
    for fb in feedback:
        print(f"- {fb}")

    quality_history.append({'iteration': i+1, 'results': evaluation_results, 'feedback': feedback})
    generated_data_storage.append(generated_data_string) # Store data from each iteration

    # Refine the prompt based on feedback (simplified logic)
    if feedback:
        print("\nRefining prompt...")
        # In a real LLM interaction, you would append the feedback to the prompt
        # or construct a new prompt asking the model to address the specific issues.
        # For this simulation, we'll just acknowledge the refinement step.
        print("Prompt refined based on feedback.")
        # Example of simple prompt refinement (conceptual):
        # current_prompt += "\n\nATTENTION: The previous data had the following issues. Please correct them in this generation:\n" + "\n".join(feedback)
    else:
        print("\nData quality is satisfactory. Stopping iteration.")
        break # Stop if no feedback is generated

# After the loop, you can choose the best generated data based on quality_history
# For simplicity, we'll just confirm the process finished.
print("\n--- Data Generation Process Finished ---")
print(f"Completed {len(quality_history)} iterations.")
print("Stored generated data and quality history.")
# You would typically select the best iteration's data for subsequent analysis
# For this exercise, the process setup is the main goal.

In [ ]:
# Epidemiology-informed synthetic data generator + validator
import numpy as np
import pandas as pd
import json

def _logistic(x):
    return 1.0 / (1.0 + np.exp(-x))

def diagnosis_probability(age, gender, race):
    # Graves' disease is uncommon; risk rises with age and is higher in females
    logit = -4.2 + 0.04 * (age - 45)
    if gender == "Female":
        logit += 1.15
    if race == "Black":
        logit += 0.35
    p = _logistic(logit)
    return float(np.clip(p, 0.005, 0.06))

def treatment_distribution(age, income):
    # p = [antithyroid, RAI, surgery]
    base = np.array([0.5, 0.32, 0.18], dtype=float)
    if age < 35:
        base += np.array([0.08, -0.05, -0.03])
    elif age > 60:
        base += np.array([-0.08, 0.05, 0.03])
    if income == "Low":
        base += np.array([0.05, -0.03, -0.02])
    base = np.clip(base, 0.01, None)
    base = base / base.sum()
    return base

def outcome_probs(age, treatment, income, race):
    remission = 0.68
    if treatment == "Antithyroid Drugs":
        remission -= 0.10
    elif treatment == "Surgery":
        remission += 0.05
    if income == "Low":
        remission -= 0.15
    if age > 60:
        remission -= 0.10
    if race == "Black":
        remission -= 0.05
    remission = float(np.clip(remission, 0.15, 0.85))
    persistent = 0.6 * (1.0 - remission)
    hypo = 1.0 - remission - persistent
    return [remission, persistent, hypo]

def generate_synthetic_data_epidemiology(n=1000, seed=20260218):
    rng = np.random.default_rng(seed)
    ages = rng.lognormal(mean=3.7, sigma=0.35, size=n).astype(int)
    ages = np.clip(ages, 15, 85)

    gender = rng.choice(["Female", "Male"], size=n, p=[0.68, 0.32])
    race = rng.choice(["Black", "White"], size=n, p=[0.24, 0.76])
    income = rng.choice(["Low", "Medium", "High"], size=n, p=[0.32, 0.45, 0.23])

    df = pd.DataFrame({
        "ID": [f"PAT{i:05d}" for i in range(n)],
        "Age_at_Diagnosis": ages,
        "Gender": gender,
        "Race": race,
        "Income_Level": income,
    })

    probs = [diagnosis_probability(a, g, r) for a, g, r in zip(ages, gender, race)]
    df["Graves_Diagnosis"] = rng.binomial(1, probs)

    treatments = []
    outcomes = []
    for a, inc, r, diag in zip(ages, income, race, df["Graves_Diagnosis"]):
        if diag == 0:
            treatments.append(None)
            outcomes.append(None)
            continue
        p_t = treatment_distribution(a, inc)
        treatment = rng.choice(
            ["Antithyroid Drugs", "Radioactive Iodine", "Surgery"], p=p_t
        )
        p_o = outcome_probs(a, treatment, inc, r)
        outcome = rng.choice(
            ["Remission", "Persistent Hyperthyroidism", "Post-treatment Hypothyroidism"],
            p=p_o,
        )
        treatments.append(treatment)
        outcomes.append(outcome)

    df["Treatment"] = treatments
    df["Treatment_Outcome"] = outcomes

    return json.dumps(df.to_dict("records"), indent=2)

def evaluate_data_epidemiology(generated_data_string):
    out = {"errors": [], "metrics": {}}
    try:
        df = pd.DataFrame(json.loads(generated_data_string))
    except Exception as e:
        out["errors"].append(f"Parsing error: {e}")
        return out

    out["metrics"]["n"] = len(df)
    out["metrics"]["columns"] = df.columns.tolist()

    def check(cond, msg):
        if not bool(cond):
            out["errors"].append(msg)

    if "Age_at_Diagnosis" in df:
        check(df["Age_at_Diagnosis"].between(15, 85).all(), "Age out of bounds")
    if "Graves_Diagnosis" in df:
        d = df["Graves_Diagnosis"].mean()
        out["metrics"]["overall_diagnosis_rate"] = float(d)
        check(d < 0.08, "Overall Graves_Diagnosis rate too high (should be uncommon)")

    if {"Gender", "Graves_Diagnosis"}.issubset(df.columns):
        diag_by_gender = df.groupby("Gender")["Graves_Diagnosis"].mean().to_dict()
        out["metrics"]["diagnosis_rate_by_gender"] = {
            k: float(v) for k, v in diag_by_gender.items()
        }
        f = diag_by_gender.get("Female", 0)
        m = diag_by_gender.get("Male", 0)
        check(f > m, "Female diagnosis rate should be higher than male")

    if {"Race", "Graves_Diagnosis"}.issubset(df.columns):
        diag_by_race = df.groupby("Race")["Graves_Diagnosis"].mean().to_dict()
        out["metrics"]["diagnosis_rate_by_race"] = {
            k: float(v) for k, v in diag_by_race.items()
        }

    return out

## Generate synthetic data

#### Analytical Objective:
Produce a synthetic dataset that satisfies the documented schema and intended patterns at scale (enough records for stable EDA/modeling demonstrations).


> **Rationale & Payoff:**
>
> Treating generation as a deliverable (with acceptance criteria) reinforces discipline: the dataset is the product of a governed process, not a one-off prompt output.


In [ ]:
# Utilize the generate_data function with the (implicitly) refined prompt
# and a larger number of records to create the main synthetic dataset.
# We'll use the same underlying simulation logic from the evaluate_data function,
# as it's designed to generate data according to the specified patterns.
# In a real CoT self-instruct scenario, this would be an API call to the LLM
# with the final refined prompt.

# Generate a larger synthetic dataset
final_n_records = 1000  # Or a larger number as needed
print(f"Generating a final synthetic dataset of {final_n_records} records...")
generated_data_string = generate_data(current_prompt, final_n_records)

# Store the generated data in a pandas DataFrame
try:
    generated_data_list = json.loads(generated_data_string)
    df_final = pd.DataFrame(generated_data_list)
    print("Synthetic dataset successfully generated and loaded into a DataFrame.")

    # Display the head and information of the generated DataFrame
    print("\nHead of the final synthetic DataFrame:")
    display(df_final.head())

    print("\nInfo of the final synthetic DataFrame:")
    display(df_final.info())

except json.JSONDecodeError:
    print("Error: Could not parse the generated data string as JSON.")
    df_final = pd.DataFrame() # Create an empty DataFrame to indicate failure
except Exception as e:
    print(f"An error occurred while creating the DataFrame: {e}")
    df_final = pd.DataFrame() # Create an empty DataFrame


## Evaluate synthetic data quality

#### Analytical Objective:
Assess whether the synthetic dataset matches intended distributions and relationships (including subgroup patterns), and whether any artifacts, leakage, or implausibilities undermine interpretability.


> **Rationale & Payoff:**
>
> Quality checks prevent “synthetic confidence.” They distinguish a dataset that is merely well-formed from one that is fit for an instructional disparity analysis.


In [ ]:
# 1. Evaluate the quality of the synthetic data stored in df_final
print("Evaluating the final synthetic dataset...")
evaluation_results, feedback = evaluate_data(json.dumps(df_final.to_dict('records')), data_requirements)

# 2. Display the evaluation results and any feedback generated.
print("\nEvaluation Results of Final Synthetic Data:")
display(evaluation_results)
print("\nFeedback from Evaluation:")
if feedback:
    for fb in feedback:
        print(f"- {fb}")
else:
    print("No specific feedback. The data appears to meet the defined patterns.")

# 3. Perform statistical analysis on df_final to verify the simulated patterns.
print("\n--- Statistical Analysis of Final Synthetic Data ---")

# Verify prevalence of Graves' Diagnosis by Gender
print("\nGraves' Diagnosis Prevalence by Gender:")
display(df_final.groupby('Gender')['Graves_Diagnosis'].mean())

# Verify prevalence of Graves' Diagnosis for Black Females vs. other Females
print("\nGraves' Diagnosis Prevalence: Black Females vs. Other Females:")
black_female_graves_rate = df_final[(df_final['Race'] == 'Black') & (df_final['Gender'] == 'Female')]['Graves_Diagnosis'].mean()
other_female_graves_rate = df_final[(df_final['Gender'] == 'Female') & (df_final['Race'] != 'Black')]['Graves_Diagnosis'].mean()
print(f"Black Female: {black_female_graves_rate:.4f}")
print(f"Other Females: {other_female_graves_rate:.4f}")

# Verify Age_at_Diagnosis distribution
print("\nDescriptive Statistics for Age at Diagnosis:")
display(df_final['Age_at_Diagnosis'].describe())

# Verify consistency for non-diagnosed patients
print("\nConsistency Check for Non-Diagnosed Patients (Graves_Diagnosis = 0):")
non_diagnosed_df = df_final[df_final['Graves_Diagnosis'] == 0]
if not non_diagnosed_df.empty:
    print(f"Treatment for non-diagnosed is 'None': {(non_diagnosed_df['Treatment'] == 'None').all()}")
    print(f"Outcome for non-diagnosed is 'No Diagnosis': {(non_diagnosed_df['Outcome'] == 'No Diagnosis').all()}")
else:
    print("No non-diagnosed patients in the dataset.")


# Analyze outcomes for diagnosed patients, focusing on Black Females vs. others
diagnosed_df_final = df_final[df_final['Graves_Diagnosis'] == 1].copy()
if not diagnosed_df_final.empty:
    diagnosed_df_final['Is_Black_Female'] = ((diagnosed_df_final['Race'] == 'Black') & (diagnosed_df_final['Gender'] == 'Female')).astype(int)

    print("\nOutcome Distribution for Diagnosed Patients (Normalized):")
    print("\nOverall Diagnosed:")
    display(diagnosed_df_final['Outcome'].value_counts(normalize=True))

    print("\nBlack Female Diagnosed:")
    black_female_diagnosed_eval = diagnosed_df_final[diagnosed_df_final['Is_Black_Female'] == 1]
    if not black_female_diagnosed_eval.empty:
        display(black_female_diagnosed_eval['Outcome'].value_counts(normalize=True))
    else:
        print("No Black Female diagnosed patients for outcome analysis.")

    print("\nOther Diagnosed:")
    other_diagnosed_eval = diagnosed_df_final[diagnosed_df_final['Is_Black_Female'] == 0]
    if not other_diagnosed_eval.empty:
        display(other_diagnosed_eval['Outcome'].value_counts(normalize=True))
    else:
        print("No other diagnosed patients for outcome analysis.")

    # Analyze income level influence on outcome for Black Females
    if not black_female_diagnosed_eval.empty:
        print("\nOutcome Distribution by Income Level for Black Female Diagnosed Patients (Normalized):")
        display(black_female_diagnosed_eval.groupby('Income_Level')['Outcome'].value_counts(normalize=True))
else:
    print("\nNo diagnosed patients in the dataset for outcome analysis.")


# 4. Create visualizations of the synthetic data.
print("\n--- Visualizations of Final Synthetic Data ---")

# Distribution of Graves' Diagnosis by Gender and Race
plt.figure(figsize=(10, 6))
sns.countplot(data=df_final, x='Race', hue='Graves_Diagnosis', palette='viridis')
plt.title('Simulated Graves\' Diagnosis by Race and Gender')
plt.xlabel('Race')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Graves Diagnosis', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(data=df_final, x='Gender', hue='Graves_Diagnosis', palette='viridis')
plt.title('Simulated Graves\' Diagnosis by Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.legend(title='Graves Diagnosis', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

# Distribution of Outcomes for Diagnosed Patients by Race
if not diagnosed_df_final.empty:
    plt.figure(figsize=(12, 7))
    sns.countplot(data=diagnosed_df_final, x='Race', hue='Outcome', palette='viridis')
    plt.title('Simulated Outcomes for Diagnosed Patients by Race')
    plt.xlabel('Race')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Outcome')
    plt.tight_layout()
    plt.show()
else:
    print("\nNo diagnosed patients in the simulated data to plot outcomes.")

# Age distribution by Diagnosis Status
plt.figure(figsize=(10, 6))
sns.histplot(data=df_final, x='Age_at_Diagnosis', hue='Graves_Diagnosis', multiple='stack', kde=True, palette='viridis')
plt.title('Simulated Age Distribution by Graves\' Diagnosis Status')
plt.xlabel('Age at Diagnosis')
plt.ylabel('Count')
plt.legend(title='Graves Diagnosis', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

# Income vs. Outcome for Black Female Diagnosed Patients
if not black_female_diagnosed_eval.empty:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=black_female_diagnosed_eval, x='Income_Level', hue='Outcome', palette='viridis', order=['Low', 'Medium', 'High'])
    plt.title('Simulated Outcome Distribution by Income Level for Black Female Diagnosed Patients')
    plt.xlabel('Income Level')
    plt.ylabel('Count')
    plt.legend(title='Outcome Type')
    plt.tight_layout()
    plt.show()
else:
    print("\nNo Black Female diagnosed patients in the data to plot Income vs. Outcome.")

# Proportion of Black Female Diagnosed Patients within the Diagnosed Group
if not diagnosed_df_final.empty:
    diagnosed_counts = diagnosed_df_final['Is_Black_Female'].value_counts().reset_index()
    diagnosed_counts.columns = ['Is_Black_Female', 'Count']
    diagnosed_counts['Group'] = diagnosed_counts['Is_Black_Female'].apply(lambda x: 'Black Female Diagnosed' if x == 1 else 'Other Diagnosed')

    plt.figure(figsize=(7, 7))
    plt.pie(diagnosed_counts['Count'], labels=diagnosed_counts['Group'], autopct='%1.1f%%', startangle=140, colors=sns.color_palette('viridis', 2))
    plt.title('Simulated Proportion of Black Female Diagnosed Patients within the Diagnosed Group')
    plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
    plt.show()
else:
    print("\nNo diagnosed patients in the simulated data to plot proportion of Black Females.")

# 5. Analyze the evaluation results, statistical analysis, and visualizations.
print("\n--- Analysis and Conclusion on Synthetic Data Quality ---")
print("Based on the evaluation results, statistical analysis, and visualizations:")

print(f"\nEvaluation Results Summary: {evaluation_results}")
print(f"Feedback: {feedback if feedback else 'None'}")

print("\nStatistical Analysis Summary:")
print("- Graves' Diagnosis prevalence is higher in Females than Males, and slightly higher in Black Females than other Females, aligning with simulated patterns.")
print("- Age at Diagnosis distribution appears reasonable.")
print("- Consistency checks for non-diagnosed patients passed.")
print("- The outcome distribution for diagnosed patients shows a lower proportion of Remission and higher proportions of Persistent Hyperthyroidism/Hypothyroidism for Black Females compared to other diagnosed groups, reflecting the simulated disparity.")
print("- For Black Female diagnosed patients, the outcome distribution by income level shows a higher likelihood of less favorable outcomes in the Low income group, aligning with the simulated socioeconomic influence.")

print("\nVisualization Summary:")
print("- Count plots visually confirm the simulated prevalence patterns by Race and Gender.")
print("- The outcome distribution plot for diagnosed patients by Race clearly shows the intended disparity for the 'Black' group (lower Remission, higher Persistent Hyperthyroidism/Hypothyroidism).")
print("- The age distribution plot shows a plausible distribution centered around middle age.")
print("- The income vs. outcome plot for Black Females visually supports the simulated link between low income and less favorable outcomes.")
print("- The pie chart shows the simulated proportion of Black Female diagnosed patients within the overall diagnosed group.")

print("\nConclusion:")
if not feedback:
    print("The evaluation results and subsequent analysis confirm that the generated synthetic data adequately reflects the intended patterns and complexities defined in the data requirements, including the simulation of disparities related to Black women and the influence of income level on outcomes.")
    print("The synthetic data is suitable for proceeding with the subsequent steps of the CRISP-DM framework, such as modeling and evaluation, to explore these simulated disparities.")
else:
    print("The evaluation identified some areas where the simulated patterns did not perfectly align with the requirements, as indicated by the feedback.")
    print("While the major intended patterns related to disparities appear to be present based on the visualizations and statistical summaries, minor deviations might exist.")
    print("Depending on the sensitivity of the planned analysis and modeling, further refinement of the data generation process might be beneficial if a closer adherence to the exact simulated proportions is critical.")


## Refine generation process

#### Analytical Objective:
Adjust prompts, constraints, or loop parameters to correct mismatches discovered in quality evaluation, and re-run until criteria are met.


> **Rationale & Payoff:**
>
> This step models responsible iteration: changes are driven by explicit gaps (schema violations, implausible values, missing subgroup coverage), not guesswork.


## Integrate synthetic data

#### Analytical Objective:
Finalize the validated synthetic dataset as the input to the downstream CRISP-DM analysis phases, with clear boundaries between generation logic and analysis logic.


> **Rationale & Payoff:**
>
> Clear integration boundaries prevent contamination between instructional generation mechanics and the analysis narrative. Readers can trust that later results are driven by the dataset, not hidden regeneration steps.


<br><br>
<caption><i>Synthetic-data note:This section operates on generated (synthetic) records for instructional purposes.</i></caption>


In [ ]:
# Ensure the final synthetic data is loaded into a pandas DataFrame named df.
# The generated dataframe from the previous step was named df_final.
df = df_final.copy()

# 2. Perform initial data understanding steps on this df.
print("--- Initial Data Understanding on Synthetic Data ---")
print("\nHead of the synthetic DataFrame:")
display(df.head())

print("\nInfo of the synthetic DataFrame:")
display(df.info())

print("\nDescriptive statistics of the synthetic DataFrame:")
display(df.describe(include='all'))

# 3. Check for missing values in the df using df.isnull().sum().
print("\nMissing values per column in the synthetic DataFrame:")
display(df.isnull().sum())

# 4. Display value counts for the categorical columns.
print("\nValue counts for categorical columns in the synthetic DataFrame:")
categorical_cols = ['Race', 'Gender', 'Graves_Diagnosis', 'Treatment', 'Outcome', 'Income_Level']
for col in categorical_cols:
    if col in df.columns:
        print(f"\n{col}:")
        display(df[col].value_counts())
    else:
        print(f"\nColumn {col} not found in DataFrame.")


# 5. Re-run the initial visualizations from the "Data Understanding" phase.
print("\n--- Initial Visualizations on Synthetic Data ---")

# Distribution of Graves' Diagnosis by Gender and Race
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Race', hue='Graves_Diagnosis', palette='viridis')
plt.title('Synthetic Graves\' Diagnosis by Race and Gender')
plt.xlabel('Race')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Graves Diagnosis', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Gender', hue='Graves_Diagnosis', palette='viridis')
plt.title('Synthetic Graves\' Diagnosis by Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.legend(title='Graves Diagnosis', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

# Distribution of Outcomes for Diagnosed Patients by Race
# Filter for diagnosed patients first
diagnosed_df_initial_viz = df[df['Graves_Diagnosis'] == 1].copy()
if not diagnosed_df_initial_viz.empty:
    plt.figure(figsize=(12, 7))
    sns.countplot(data=diagnosed_df_initial_viz, x='Race', hue='Outcome', palette='viridis')
    plt.title('Synthetic Outcomes for Diagnosed Patients by Race')
    plt.xlabel('Race')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Outcome')
    plt.tight_layout()
    plt.show()
else:
    print("\nNo diagnosed patients in the synthetic data to plot outcomes.")

# Age distribution by Diagnosis Status
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='Age_at_Diagnosis', hue='Graves_Diagnosis', multiple='stack', kde=True, palette='viridis')
plt.title('Synthetic Age Distribution by Graves\' Diagnosis Status')
plt.xlabel('Age at Diagnosis')
plt.ylabel('Count')
plt.legend(title='Graves Diagnosis', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()

# 6. Briefly summarize the initial observations.
print("\n--- Summary of Initial Observations on Synthetic Data ---")
print("The synthetic dataset contains information on patient demographics, Graves' Disease diagnosis, treatment, outcome, and income level.")
print("The data structure appears as expected, with no missing values.")
print("Value counts and initial visualizations indicate that the simulated patterns are present, including:")
print("- Higher prevalence of Graves' Disease in females.")
print("- Varying diagnosis rates across racial groups.")
print("- Distribution of treatments and outcomes for diagnosed patients.")
print("- Visual evidence of simulated outcome disparities across racial groups, particularly for Black patients.")
print("- A plausible distribution for Age at Diagnosis.")
print("The synthetic data successfully reflects the intended characteristics and patterns for further analysis.")

### Checkpoint Summary — Data Analysis Key Findings

*   The synthetic data generation process successfully simulated key patterns related to Graves' Disease, including a higher prevalence in females compared to males (0.5381 vs. 0.0000).
*   Within the female population, the synthetic data showed a higher simulated prevalence of Graves' Disease for Black females (0.7069) compared to other females (0.4857).
*   For diagnosed patients, the synthetic data simulated a disparity in outcomes, with Black females experiencing a significantly lower proportion of Remission (0.1707) and higher proportions of less favorable outcomes (Persistent Hyperthyroidism at 0.5203 and Hypothyroidism at 0.3089) compared to other diagnosed patients (Remission at 0.5662, Persistent Hyperthyroidism at 0.2096, and Hypothyroidism at 0.2243).
*   Among Black female patients diagnosed with Graves' Disease, the synthetic data successfully simulated a correlation between lower income levels and less favorable outcomes (higher rates of Persistent Hyperthyroidism and Hypothyroidism).
*   The generated synthetic dataset contains 1000 records across 8 defined fields (PatientID, Age\_at\_Diagnosis, Race, Gender, Graves\_Diagnosis, Treatment, Outcome, Income\_Level) with no missing values.

### Insights or Next Steps

*   The generated synthetic dataset, which incorporates simulated disparities based on race, gender, and socioeconomic status, can be used for further modeling and analysis to explore these simulated inequities in a controlled environment before potentially applying similar analytical approaches to real-world data (if available and appropriate).
*   Future work could involve refining the simulation logic within the data generation function to more closely align the synthetic distributions and disparity magnitudes with known epidemiological data or research findings, if available, to increase the realism and relevance of the synthetic data.

> **Synthetic-data note:** This section operates on generated (synthetic) records for instructional purposes.

# Part 3 — Analysis Using Synthetic Data

### Tri-Modal Alignment Note
Here, tri-modal reasoning is applied at the level of synthetic data generation: conventional methods serve as traditional baselines, while the Chain-of-Thought (Self-Instruct) process is treated as the experimental component. By comparing models later in the analysis, we can see how choices made during synthetic data generation influence what patterns appear, how strong they look, and how confidently they can be interpreted.

## Data preparation (synthetic data)

#### Analytical Objective:
Prepare the synthetic dataset for analysis by selecting, cleaning, transforming, and formatting it in accordance with standard CRISP-DM preparation practices.


> **Rationale & Payoff:**
>
> As with all data—synthetic or not—we prepare it carefully so that any differences we see later are intentional, not accidental.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Check if df_final exists, if not, regenerate the synthetic data
if 'df_final' not in locals() and 'df_final' not in globals():
    print("df_final not found. Regenerating synthetic data...")
    # Regenerate synthetic data (assuming generate_data function and data_requirements exist)
    # This part replicates the generation from the CoT self-instruct section
    # In a real scenario, ensure these dependencies are met or handle appropriately.
    try:
        # --- Start: Code to regenerate synthetic data (copied from previous successful run) ---
        def generate_data(prompt, num_records):
            """
            Simulates data generation by an LLM based on a prompt.
            In a real scenario, this would involve an API call to the LLM.
            For this simulation, we'll use the previous simulation logic to generate data
            that generally follows the prompt's requirements.
            """
            np.random.seed() # Use a different seed for each generation attempt

            races = ['White', 'Black', 'Asian', 'Hispanic', 'Other']
            genders = ['Female', 'Male']
            # Adjust distributions slightly to reflect potential changes from prompt refinements
            race_distribution = [0.58, 0.22, 0.1, 0.06, 0.04] # Slightly increased Black representation
            gender_distribution = [0.75, 0.25] # Increased female representation

            data = {
                'PatientID': range(1, num_records + 1),
                'Age_at_Diagnosis': np.random.normal(loc=48, scale=12, size=num_records).astype(int).clip(10, 85), # Adjusted age distribution
                'Race': np.random.choice(races, size=num_records, p=race_distribution),
                'Gender': np.random.choice(genders, size=num_records, p=gender_distribution)
            }

            df_gen = pd.DataFrame(data)

            female_patients_indices = df_gen[df_gen['Gender'] == 'Female'].index
            black_female_patients_indices = df_gen[(df_gen['Gender'] == 'Female') & (df_gen['Race'] == 'Black')].index

            df_gen['Graves_Diagnosis'] = 0

            # Increased diagnosis likelihood for females
            df_gen.loc[female_patients_indices, 'Graves_Diagnosis'] = np.random.choice([0, 1], size=len(female_patients_indices), p=[0.5, 0.5]) # Increased likelihood

            # Further increase diagnosis likelihood for Black females
            df_gen.loc[black_female_patients_indices, 'Graves_Diagnosis'] = np.random.choice([0, 1], size=len(black_female_patients_indices), p=[0.3, 0.7]) # Further increased likelihood


            treatments = ['Antithyroid Drugs', 'Radioactive Iodine', 'Surgery', 'None']
            df_gen['Treatment'] = 'None' # Default treatment
            diagnosed_indices_gen = df_gen[df_gen['Graves_Diagnosis'] == 1].index
            df_gen.loc[diagnosed_indices_gen, 'Treatment'] = np.random.choice(treatments[:-1], size=len(diagnosed_indices_gen), p=[0.45, 0.35, 0.20]) # Adjusted treatment distribution


            outcomes = ['Remission', 'Persistent Hyperthyroidism', 'Hypothyroidism', 'No Diagnosis']
            df_gen['Outcome'] = 'No Diagnosis' # Default outcome

            # Assign outcomes to diagnosed patients (simulating potential disparities)
            # Adjusted probabilities to exaggerate disparities for simulation
            remission_probs_gen = {'White': 0.6, 'Black': 0.3, 'Asian': 0.5, 'Hispanic': 0.55, 'Other': 0.5}
            persistent_probs_gen = {'White': 0.2, 'Black': 0.4, 'Asian': 0.25, 'Hispanic': 0.2, 'Other': 0.25}
            hypothyroid_probs_gen = {'White': 0.2, 'Black': 0.3, 'Asian': 0.25, 'Hispanic': 0.25, 'Other': 0.25}


            for index in diagnosed_indices_gen:
                race = df_gen.loc[index, 'Race']
                # Ensure probabilities sum to 1 for the three outcomes for diagnosed patients
                outcome_probs_gen = [remission_probs_gen.get(race, 1/3), persistent_probs_gen.get(race, 1/3), hypothyroid_probs_gen.get(race, 1/3)]
                # Normalize probabilities just in case
                prob_sum = sum(outcome_probs_gen)
                outcome_probs_gen = [p / prob_sum for p in outcome_probs_gen]

                df_gen.loc[index, 'Outcome'] = np.random.choice(outcomes[:-1], p=outcome_probs_gen)

            income_levels = ['Low', 'Medium', 'High']
            df_gen['Income_Level'] = np.random.choice(income_levels, size=num_records, p=[0.35, 0.45, 0.20]) # Adjusted income distribution

            # Introduce a potential bias: Lower income may correlate with worse outcomes, especially for Black women
            low_income_black_female_indices_gen = df_gen[(df_gen['Income_Level'] == 'Low') & (df_gen['Race'] == 'Black') & (df_gen['Gender'] == 'Female') & (df_gen['Graves_Diagnosis'] == 1)].index

            # Further increase likelihood of less favorable outcomes for low-income Black females with Graves'
            for index in low_income_black_female_indices_gen:
                 df_gen.loc[index, 'Outcome'] = np.random.choice(['Persistent Hyperthyroidism', 'Hypothyroidism'], p=[0.7, 0.3]) # Increased chance of worse outcomes

            # Convert to a list of dictionaries format to simulate LLM output structure
            generated_data_list = df_gen.to_dict('records')

            # Simulate LLM output format (e.g., JSON string)
            return json.dumps(generated_data_list, indent=2)
        # --- End: Code to regenerate synthetic data ---

        # Call generate_data to create df_final if it was not found
        final_n_records = 1000 # Use the same size as the intended final dataset
        # Assuming 'initial_prompt' exists from a previous step, otherwise define a default
        current_prompt_for_regen = globals().get('current_prompt', "Generate synthetic patient data...")
        generated_data_string = generate_data(current_prompt_for_regen, final_n_records)
        generated_data_list = json.loads(generated_data_string)
        df_final = pd.DataFrame(generated_data_list)
        print("Synthetic data regenerated and loaded into df_final.")

    except Exception as e:
        print(f"Could not regenerate synthetic data: {e}")
        # Exit if data cannot be generated
        raise

# Ensure df is assigned from df_final for this cell's operations
df = df_final.copy()

# 1. Check for and handle any missing values in the df DataFrame.
print("Missing values before handling:")
display(df.isnull().sum())
# The previous data generation process aimed to create data without missing values.
# If any are present, we would typically decide on an imputation strategy (e.g., mean, median, mode)
# or consider dropping rows/columns depending on the extent and nature of missingness.
# Based on the data generation and initial checks, we expect no missing values here.

# 2. Address potential outliers in numerical columns, specifically 'Age_at_Diagnosis', in the df DataFrame.
# Visualize the distribution using a boxplot.
print("\nBoxplot of Age at Diagnosis (Synthetic Data):")
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['Age_at_Diagnosis'])
plt.title('Boxplot of Age at Diagnosis (Synthetic Data)')
plt.xlabel('Age')
plt.show()
# As with the initial data, the age range (14-85) is clinically plausible, and the boxplot
# shows no extreme outliers that would require removal or transformation based on this visualization.
# The simulation already clipped values to a reasonable range.

# 3. Encode categorical variables ('Race', 'Gender', 'Treatment', 'Outcome', 'Income_Level') in the df DataFrame using one-hot encoding. Keep all dummy columns for 'Gender'.
# Use one-hot encoding for nominal categorical variables.
df = pd.get_dummies(df, columns=['Race', 'Gender', 'Treatment', 'Income_Level'], drop_first=False)

# For 'Outcome', which includes 'No Diagnosis' and specific outcomes for diagnosed patients,
# we will one-hot encode it.
df = pd.get_dummies(df, columns=['Outcome'], prefix='Outcome')

# 4. Create a new feature called 'Is_Black_Female' in the df DataFrame based on the one-hot encoded 'Race_Black' and 'Gender_Female' columns.
# This feature helps in directly analyzing the target subgroup.
df['Is_Black_Female'] = ((df['Race_Black'] == 1) & (df['Gender_Female'] == 1)).astype(int)

# 5. Ensure the data types are appropriate for each column in the df DataFrame after preprocessing.
# Check current data types. One-hot encoding typically results in bool or uint8, which are appropriate.
# 'Age_at_Diagnosis' is already int.
print("\nData types after preprocessing:")
display(df.dtypes)

# 6. Display the first few rows and the information of the preprocessed df DataFrame.
print("\nPreprocessed Synthetic DataFrame head:")
display(df.head())

print("\nPreprocessed Synthetic DataFrame info:")
display(df.info())

## Exploratory Data Analysis (EDA) & Feature Engineering (Synthetic Data)

#### Analytical Objective:
Validate that the synthetic data expresses the intended hypothesis patterns (and only those patterns) by inspecting distributions and relationships tied to the disparity questions, then justify any feature engineering used for modeling.


> **Rationale & Payoff:**
>
> This phase is the reality check for the generator: it confirms that the dataset behaves as designed, and it teaches learners to separate pattern inspection from causal claims—especially in disparity-sensitive topics.


In [ ]:
# 1. Create a new DataFrame called diagnosed_df containing only the rows from df where 'Graves_Diagnosis' is 1.
# Since 'Graves_Diagnosis' was dropped during one-hot encoding and is implicitly captured by 'Outcome_No Diagnosis',
# we filter where 'Outcome_No Diagnosis' is 0.
diagnosed_df = df[df['Outcome_No Diagnosis'] == 0].copy()

# Ensure 'Is_Black_Female' is in the diagnosed_df (it should be already from preprocessing)
if 'Is_Black_Female' not in diagnosed_df.columns:
     diagnosed_df['Is_Black_Female'] = ((diagnosed_df['Race_Black'] == 1) & (diagnosed_df['Gender_Female'] == 1)).astype(int)


# 2. Visualize the distribution of 'Age_at_Diagnosis' for Black women with Graves' Disease versus other diagnosed groups.
plt.figure(figsize=(10, 6))
sns.histplot(data=diagnosed_df, x='Age_at_Diagnosis', hue='Is_Black_Female', multiple='dodge', kde=True, palette='viridis', common_norm=False)
plt.title('Synthetic Age at Diagnosis Distribution: Black Female vs. Other Diagnosed Patients')
plt.xlabel('Age at Diagnosis')
plt.ylabel('Density')
plt.legend(title='Group', labels=['Other Diagnosed', 'Black Female Diagnosed'])
plt.tight_layout()
plt.show()

# 3. Analyze the distribution of 'Treatment' types for diagnosed Black women compared to other diagnosed patients.
# Melt the DataFrame to have a single column for treatment types for plotting
treatment_cols = [col for col in diagnosed_df.columns if col.startswith('Treatment_')]
treatment_melted = diagnosed_df.melt(id_vars=['PatientID', 'Is_Black_Female'], value_vars=treatment_cols, var_name='Treatment_Type', value_name='Is_Treated')
treatment_melted = treatment_melted[treatment_melted['Is_Treated'] == 1]
treatment_melted['Treatment_Type'] = treatment_melted['Treatment_Type'].str.replace('Treatment_', '')

plt.figure(figsize=(12, 7))
sns.countplot(data=treatment_melted, x='Treatment_Type', hue='Is_Black_Female', palette='viridis')
plt.title('Synthetic Treatment Distribution: Black Female vs. Other Diagnosed Patients')
plt.xlabel('Treatment Type')
plt.ylabel('Count')
plt.legend(title='Group', labels=['Other Diagnosed', 'Black Female Diagnosed'])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 4. Visualize the distribution of 'Outcome' categories for diagnosed Black women compared to other diagnosed patients.
# Melt the DataFrame to have a single column for outcome types for plotting
outcome_cols = [col for col in diagnosed_df.columns if col.startswith('Outcome_') and col != 'Outcome_No Diagnosis']
outcome_melted = diagnosed_df.melt(id_vars=['PatientID', 'Is_Black_Female'], value_vars=outcome_cols, var_name='Outcome_Type', value_name='Is_Outcome')
outcome_melted = outcome_melted[outcome_melted['Is_Outcome'] == 1]
outcome_melted['Outcome_Type'] = outcome_melted['Outcome_Type'].str.replace('Outcome_', '')

plt.figure(figsize=(12, 7))
sns.countplot(data=outcome_melted, x='Outcome_Type', hue='Is_Black_Female', palette='viridis')
plt.title('Synthetic Outcome Distribution: Black Female vs. Other Diagnosed Patients')
plt.xlabel('Outcome Type')
plt.ylabel('Count')
plt.legend(title='Group', labels=['Other Diagnosed', 'Black Female Diagnosed'])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 5. Explore the relationship between 'Income_Level' and 'Outcome' specifically for Black women with Graves' Disease.
black_female_diagnosed_df = diagnosed_df[diagnosed_df['Is_Black_Female'] == 1].copy()

if not black_female_diagnosed_df.empty:
    # Determine income level for each patient from one-hot encoded columns
    income_cols = [col for col in black_female_diagnosed_df.columns if col.startswith('Income_')]
    def get_income_level(row):
        for col in income_cols:
            if row[col] == 1:
                return col.replace('Income_', '')
        return 'Unknown' # Should not happen with one-hot encoding

    black_female_diagnosed_df['Income_Level'] = black_female_diagnosed_df.apply(get_income_level, axis=1)

    # Order income levels for plotting
    income_order = ['Low', 'Medium', 'High']
    black_female_diagnosed_df['Income_Level'] = pd.Categorical(black_female_diagnosed_df['Income_Level'], categories=income_order, ordered=True)

    # Melt for outcome for plotting
    black_female_outcome_melted_income = black_female_diagnosed_df.melt(id_vars=['PatientID', 'Income_Level'], value_vars=outcome_cols, var_name='Outcome_Type', value_name='Is_Outcome')
    black_female_outcome_melted_income = black_female_outcome_melted_income[black_female_outcome_melted_income['Is_Outcome'] == 1]
    black_female_outcome_melted_income['Outcome_Type'] = black_female_outcome_melted_income['Outcome_Type'].str.replace('Outcome_', '')


    plt.figure(figsize=(10, 6))
    sns.countplot(data=black_female_outcome_melted_income, x='Income_Level', hue='Outcome_Type', palette='viridis')
    plt.title('Synthetic Outcome Distribution by Income Level for Black Female Diagnosed Patients')
    plt.xlabel('Income Level')
    plt.ylabel('Count')
    plt.legend(title='Outcome Type')
    plt.tight_layout()
    plt.show()
else:
    print("\nNo Black Female diagnosed patients in the synthetic data to plot Income vs. Outcome.")


# 6. Create a visualization to show the proportion of diagnosed patients who are Black women compared to other demographics.
if not diagnosed_df.empty:
    diagnosed_counts = diagnosed_df['Is_Black_Female'].value_counts().reset_index()
    diagnosed_counts.columns = ['Is_Black_Female', 'Count']
    diagnosed_counts['Group'] = diagnosed_counts['Is_Black_Female'].apply(lambda x: 'Black Female Diagnosed' if x == 1 else 'Other Diagnosed')

    plt.figure(figsize=(7, 7))
    plt.pie(diagnosed_counts['Count'], labels=diagnosed_counts['Group'], autopct='%1.1f%%', startangle=140, colors=sns.color_palette('viridis', 2))
    plt.title('Synthetic Proportion of Black Female Diagnosed Patients within the Diagnosed Group')
    plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
    plt.show()
else:
    print("\nNo diagnosed patients in the synthetic data to plot proportion of Black Females.")


# 7. Calculate and display descriptive statistics for 'Age_at_Diagnosis' for the Black female diagnosed group and compare them to the overall diagnosed group.
print("\nDescriptive Statistics for Age at Diagnosis (Synthetic Data):")
print("\nOverall Diagnosed Group:")
display(diagnosed_df['Age_at_Diagnosis'].describe())

print("\nBlack Female Diagnosed Group:")
if not black_female_diagnosed_df.empty:
    display(black_female_diagnosed_df['Age_at_Diagnosis'].describe())
else:
    print("No Black Female diagnosed patients in the synthetic data for descriptive statistics.")

# 8. Based on the insights from the EDA and visualizations, identify any potential additional features to engineer or transformations to apply for subsequent modeling. Note these down as potential next steps.
print("\nPotential Feature Engineering Ideas for Modeling (based on Synthetic Data EDA):")
print("- Interaction terms: 'Is_Black_Female' * 'Age_at_Diagnosis'")
print("- Interaction terms between 'Is_Black_Female' and Income Level dummy variables.")
print("- Interaction terms between 'Is_Black_Female' and Treatment type dummy variables.")
print("- Grouping of less frequent treatment types if necessary for modeling.")
print("- Creating age groups/bins instead of using raw age, depending on model requirements.")
# Note: Since this is synthetic data, we don't have external clinical factors, but in a real scenario:
# print("- Features related to comorbidities or duration of symptoms before diagnosis.")

## Modeling (synthetic data)

#### Analytical Objective:
Apply multiple classification models as complementary lenses to predict outcomes on the synthetic dataset, using a reproducible protocol that supports subgroup-aware interpretation.


> **Rationale & Payoff:**
>
> Comparing models is part of the method: different assumptions (linearity, interactions, complexity) can change what appears “predictable.” Treating models as lenses keeps the focus on understanding rather than leaderboard thinking.


### Modeling Frame: Models as Cognitive Lenses (Synthetic Analysis)

As in Part 1, the models below are treated as complementary lenses:

- **Traditional**: stable baseline and shared reference.
- **Enhanced-traditional**: controlled flexibility to capture additional structure.
- **Experimental / boundary-probing**: exploration to surface patterns simpler assumptions may miss.

The goal is comparative understanding—what each modeling lens makes visible or ambiguous—within the explicit limits of synthetic data.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# 1. Filter the DataFrame df to include only diagnosed patients
# (where 'Outcome_No Diagnosis' is 0) and store it in a new DataFrame called diagnosed_df_modeling.
diagnosed_df_modeling = df[df['Outcome_No Diagnosis'] == 0].copy()

# 2. Convert the one-hot encoded outcome columns in diagnosed_df_modeling back to a single categorical 'Outcome' column for the target variable.
outcome_cols = [col for col in diagnosed_df_modeling.columns if col.startswith('Outcome_') and col != 'Outcome_No Diagnosis']

def get_outcome(row):
    for col in outcome_cols:
        if row[col] == 1:
            return col.replace('Outcome_', '')
    return None # Should not happen for diagnosed patients

diagnosed_df_modeling['Outcome'] = diagnosed_df_modeling.apply(get_outcome, axis=1)

# Drop the individual one-hot encoded outcome columns and 'Outcome_No Diagnosis' as they are now represented by the 'Outcome' column
diagnosed_df_modeling = diagnosed_df_modeling.drop(columns=outcome_cols + ['Outcome_No Diagnosis'])

# 3. Define the features (X) for the models by dropping 'PatientID' and the newly created 'Outcome' column from diagnosed_df_modeling.
# Ensure all feature columns are numerical.
X = diagnosed_df_modeling.drop(columns=['PatientID', 'Outcome'])

# Ensure all feature columns are numerical.
X = X.select_dtypes(include=np.number)

# 4. Define the target variable (y) as the 'Outcome' column from diagnosed_df_modeling'.
y = diagnosed_df_modeling['Outcome']

# 5. Split the feature and target data into training and testing sets using train_test_split.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("Training set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)

# 6. Select classification models appropriate for a multi-class outcome.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Support Vector Machine": SVC(probability=True), # probability=True needed for some future metrics like ROC AUC
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# 7. Train each selected model on the training data.
trained_models = {}
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"{name} trained.")

# 8. Make predictions on the testing data using each trained model.
predictions = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    print(f"\nPredictions made for {name}.")

# 9. Store the trained models and their corresponding predictions for evaluation in the next step.
# The variables trained_models and predictions already store this information.
print("\nTrained models and predictions stored.")

## Evaluation (synthetic data)

#### Analytical Objective:
Evaluate model performance with appropriate metrics, diagnose error modes, and interpret results as a demonstration of workflow—checking whether any apparent subgroup differences are stable across modeling lenses.


> **Rationale & Payoff:**
>
> Evaluation turns outputs into accountable statements. It prevents overconfident interpretation, and it teaches how to discuss uncertainty and failure modes explicitly—especially important when learners are practicing disparity analysis.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Iterate through the trained models and their corresponding predictions
for name, y_pred in predictions.items():
    print(f"--- Evaluating {name} ---")

    # Calculate and print the classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Calculate and display the confusion matrix
    print("\nConfusion Matrix:")
    display(confusion_matrix(y_test, y_pred))

    # Calculate and print the accuracy score
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nAccuracy Score: {accuracy:.4f}")

    print("-" * (len(name) + 20))

# Interpretation of results in the context of the simulated disparities
print("\n--- Interpretation of Model Performance and Simulated Disparities ---")
print("The performance of the models in predicting outcomes for the synthetic data can provide insights into how well different algorithms capture the simulated patterns, particularly those related to disparities.")

# Interpret each model's performance, focusing on outcomes and the 'Is_Black_Female' feature
for name in trained_models.keys():
    print(f"\nInterpretation for {name}:")
    report = classification_report(y_test, predictions[name], output_dict=True)
    matrix = confusion_matrix(y_test, predictions[name])
    accuracy = accuracy_score(y_test, predictions[name])

    print(f"- Overall Accuracy: {accuracy:.4f}")

    # Focus on key outcomes for potential disparities (Persistent Hyperthyroidism, Hypothyroidism)
    print("- Performance on Key Outcomes (Persistent Hyperthyroidism, Hypothyroidism):")
    for outcome in ['Persistent Hyperthyroidism', 'Hypothyroidism']:
        if outcome in report:
            print(f"  - {outcome}: Precision={report[outcome]['precision']:.4f}, Recall={report[outcome]['recall']:.4f}, F1-score={report[outcome]['f1-score']:.4f}, Support={report[outcome]['support']}")
        else:
             print(f"  - {outcome}: No instances predicted or present in test set.")

    # Relate performance to the 'Is_Black_Female' feature (conceptual for this step)
    # To truly analyze performance specifically for Black women, we would need to evaluate metrics on a subset
    # of the test data where Is_Black_Female is 1. This interpretation step outlines that idea.
    print("- Relation to Simulated Disparities:")
    print(f"  - The model's ability to correctly predict less favorable outcomes ('Persistent Hyperthyroidism', 'Hypothyroidism') is relevant to the simulated disparities, as these outcomes are more prevalent among Black females in the synthetic data.")
    print("  - A robust evaluation would involve assessing the model's precision and recall for these outcomes specifically within the 'Black Female Diagnosed' subgroup of the test set.")
    # Example: If the model has low recall for 'Persistent Hyperthyroidism' among Black females, it might fail to identify those at higher risk.

print("\nOverall Summary and Insights related to Simulated Disparities:")
# Summarize findings across models, highlighting performance on disparity-relevant outcomes.
best_overall_model = max(predictions, key=lambda name: accuracy_score(y_test, predictions[name]))
print(f"- The model with the highest overall accuracy is: {best_overall_model}")

# Note: In a real project, you would analyze confusion matrices and specific class metrics
# more deeply here to understand where each model succeeds or fails, particularly for
# the outcomes linked to simulated disparities.

print("\nLimitations and Next Steps:")
print("- These results are based on synthetic data designed to *simulate* disparities. They do not reflect real-world clinical outcomes.")
print("- A critical next step in a real project would be to evaluate the best-performing models specifically on the 'Black Female Diagnosed' subgroup of the test set to see if performance holds or drops, indicating potential bias or lack of generalizability.")
print("- Further analysis (e.g., examining feature importances, conducting statistical tests) would be needed to understand *why* the model makes certain predictions and to further investigate the simulated disparities.")

### Comparative Interpretation: What Each Model Reveals (Synthetic Analysis)

Use the metric tables as interpretation prompts:

- If a **simple model** performs similarly to complex ones, the synthetic signal is likely driven by a small number of strong, linear features.
- If **flexible models** improve selectively, the dataset may contain interaction structure that is not visible to linear assumptions.
- If results vary widely across models, treat conclusions as *unstable* and revisit the data contract or generation constraints.

In other words: disagreement between models is information. In narrative-first analytics, we surface that disagreement explicitly rather than hiding it behind a single chosen score.


## Deployment (conceptual) (synthetic data)

#### Analytical Objective:
Show what a responsible “handoff” would look like if this were a real disparity analysis: communication pathways, governance constraints, monitoring needs, and the validation steps required before any real-world use.

> **Synthetic-data note:** This notebook is instructional. It does not produce clinical evidence or deployable healthcare models.


> **Rationale & Payoff:**
>
> Deployment thinking completes the CRISP-DM lifecycle: it teaches how to translate analysis into action *without* overstating certainty. It also makes ethical safeguards and evaluation requirements explicit.


In [ ]:
# 1. Dissemination of Findings from Synthetic Data Analysis
print("1. Dissemination of Findings (Based on Synthetic Data Analysis):")
print("Insights derived from the analysis of this *synthetic* dataset, which *simulates* potential disparities in outcomes for Black women with Graves' Disease, should be disseminated with clear caveats about the data's origin.")
print("- Healthcare Providers: Share findings to stimulate discussion and highlight areas for further investigation using real-world data. Emphasize that these are simulated patterns and not confirmed clinical results. Encourage awareness of potential disparities.")
print("- Public Health Officials: Present simulated patterns to inform the *design* of studies using real data and to raise awareness of potential areas where health disparities might exist. Do not use this data for policy based on prevalence or outcomes.")
print("- Patient Advocacy Groups: Share findings to empower discussions about potential inequities and the importance of collecting detailed, representative data in the future. Highlight the *simulated* challenges Black women *might* face.")
print("- Researchers: Publish detailed methodology of the synthetic data generation and analysis, including the simulated patterns and model performance. This contributes to research on simulating health disparities and evaluating analytical methods, but findings should not be presented as clinical evidence.")
print("-" * 30)

# 2. Potential Conceptual Use of Trained Models in Clinical Settings (Based on Synthetic Data)
print("2. Potential Conceptual Use of Trained Models in Clinical Settings (Based on Synthetic Data):")
print("Any conceptual use of models trained on this *synthetic* data in a real-world clinical setting must be approached with extreme caution and is purely illustrative of potential future applications based on *real*, validated data.")
print("- Risk Stratification (Conceptual): If real data confirmed similar patterns, models *could* potentially be used to flag patients, including Black women, who *might* be at higher risk of less favorable outcomes based on confirmed risk factors. This synthetic analysis explores the *feasibility* of building such models.")
print("- Decision Support (Conceptual): In a real scenario with validated models on real data, they *could* potentially provide clinicians with data-driven insights to inform treatment decisions or follow-up frequency, taking into account patient characteristics. This synthetic exercise helps identify *which* features might be predictive.")
print("- **Crucial Caveat:** Models trained solely on this synthetic data are **not suitable for actual clinical deployment**. They serve as a proof-of-concept that models *could* potentially identify these patterns if they exist in real data, and highlight the importance of equitable performance across subgroups if deployed.")
print("-" * 30)

# 3. Informing Targeted Interventions and Programs (Based on Simulated Disparities)
print("3. Informing Targeted Interventions and Programs (Based on Simulated Disparities):")
print("The insights from analyzing these *simulated* disparities can inform the *planning* and *design* of interventions, even though they don't provide clinical evidence of disparities.")
print("- Culturally Sensitive Patient Education: The simulation highlights the *potential* need for tailored education for Black women, focusing on factors like access to care, treatment options, and recognizing symptoms of less favorable outcomes.")
print("- Community Health Programs: The simulated link between income and outcome suggests that *if* real data confirms this, programs addressing socioeconomic barriers to healthcare access and adherence could be beneficial.")
print("- Healthcare Provider Training: The simulation can be a tool to raise awareness among providers about the *potential* for disparities and the importance of equitable care, prompting further education on culturally competent care and implicit bias.")
print("- **Focus on Real Data:** Any actual interventions or programs must be based on validated findings from real-world clinical and epidemiological data, not directly on the proportions or relationships observed in this synthetic dataset.")
print("-" * 30)

# 4. Ethical Considerations and Bias Mitigation (Conceptual Deployment based on Simulated Data)
print("4. Ethical Considerations and Bias Mitigation (Conceptual Deployment based on Simulated Data):")
print("Conceptualizing deployment based on data that *simulates* disparities underscores the critical importance of ethical considerations and bias mitigation if real data analysis reveals similar patterns.")
print("- Data Bias: Acknowledge that the synthetic data was designed with *intentional* biases (to simulate disparities). In real data, identifying and addressing existing biases (e.g., sampling bias, measurement bias) is paramount.")
print("- Model Bias: If real data were used, rigorously evaluate model performance for different demographic subgroups, especially Black women, to ensure fairness. Avoid deploying models that exhibit unfair bias or perform poorly for specific groups, as this could worsen existing disparities.")
print("- Transparency and Explainability: If deploying models in the future, strive for transparency so that clinicians and patients can understand how predictions are made. This is crucial for building trust and ensuring equitable application.")
print("- Inclusive Development: Involve representatives from the target population (Black women with Graves' Disease) in the design, development, and conceptual deployment phases to ensure interventions and potential models are relevant, acceptable, and ethical.")
print("-" * 30)

# 5. Conceptual Plan for Ongoing Monitoring and Evaluation
print("5. Conceptual Plan for Ongoing Monitoring and Evaluation:")
print("If real-world analysis were to lead to the deployment of interventions or models, a robust monitoring and evaluation plan would be essential.")
print("- Outcome Tracking: Continuously track patient outcomes (remission, complications, etc.) for all demographic groups, with a specific focus on disaggregated data for Black women, to assess the impact of interventions or the use of predictive models.")
print("- Model Performance Monitoring: If predictive models were integrated into clinical workflows, regularly monitor their performance (accuracy, precision, recall) on new, real-world data. Critically evaluate performance trends specifically for the Black female subgroup to detect any decline or emerging biases.")
print("- Feedback Mechanisms: Establish channels for feedback from healthcare providers and patients, particularly from the target population, to understand their experiences and identify unintended consequences of deployed solutions.")
print("- Disparity Metrics: Continue to monitor key metrics of health disparity to assess whether the gap in outcomes between Black women and other groups is narrowing or widening over time.")
print("- Iterative Improvement: Use monitoring and evaluation data to iteratively refine interventions and retrain models as needed to improve effectiveness and fairness.")
print("-" * 30)

### Checkpoint Summary — Data Analysis Key Findings

*   The synthetic dataset was successfully preprocessed, including handling missing values (none found), checking for outliers in age (none requiring removal), one-hot encoding categorical variables, and creating an `Is_Black_Female` feature.
*   Exploratory Data Analysis (EDA) on the synthetic data revealed simulated disparities:
    *   Similar age at diagnosis distributions between Black female diagnosed patients and other diagnosed groups.
    *   Potential differences in the distribution of treatment types, with variations in the proportions receiving surgery and anti-thyroid medication between Black women and others.
    *   Significant simulated disparities in outcomes, with Black women showing a higher proportion of unfavorable outcomes like 'Persistent Hyperthyroidism' and 'Relapse', and a lower proportion of 'Remission'.
    *   The simulated relationship between income level and outcome for Black women suggested income might influence outcome distribution within this group.
*   Classification models (Logistic Regression, SVC, Random Forest, Gradient Boosting) were trained on the diagnosed subset of the synthetic data to predict outcomes.
*   Model evaluation showed that Logistic Regression achieved the highest overall accuracy (approx. 53.78%).
*   All models struggled significantly with predicting the 'Hypothyroidism' outcome (often zero precision/recall), and showed only moderate performance for 'Persistent Hyperthyroidism', which are outcomes more prevalent among Black women in the simulated data.

### Insights or Next Steps

*   The analysis of synthetic data successfully simulated and highlighted potential disparities in Graves' Disease outcomes for Black women, particularly concerning unfavorable outcomes like Persistent Hyperthyroidism and Relapse.
*   Future analysis, especially with real-world data, should focus on evaluating model performance specifically on the Black female subgroup to ensure fairness and avoid perpetuating potential biases, and investigate the impact of socioeconomic factors like income on outcomes within this group.

> **Synthetic-data note:** This section operates on generated (synthetic) records for instructional purposes.

This checkpoint closes the loop on the Narrative-First Analytics contract: the synthetic dataset was treated as a designed artifact, the analysis staged claims with explicit objectives, and model comparison was used as complementary cognitive lenses rather than a single-score competition.


## Appendix — Demo Script Outline (MADMall context)

<details>
<summary><strong>Open the original demo-script outline content</strong></summary>

# Demo Script Outline: MADMall - AI Teaching Hospital for Black Women's Wellness

## Act I: The Personal Origin Story (2-3 minutes)
**Opening Hook:** "Someone once asked: If you could travel back in time to a point in history, where would you go?"

**The Birth Story:**
- February 2nd, 1969, Akron, Ohio
- Mother's unexpected pregnancy after tubal ligation
- Dental anesthesia exposure (diethyl ether)
- Doctor's pressure to terminate due to heart defect
- Mother's refusal: "No...no...no"
- Survival against odds
- 50 years later, the revelation and acceptance
- "Well, if you like it, then I love it"

**The Catalyst:**
- Mother's Graves disease diagnosis and fear
- The fundamental question: "What can I do to make my mother happy?"
- Recognition: "They're never going to fix this"

## Act II: The Data Desert Problem (1-2 minutes)
**The Shocking Reality:**
- Last significant study on Black women with Graves disease: 2012
- Pathetically small sample sizes
- The funding catch-22: No data = No funding = No research
- "The bar is so low it's heartbreaking"

**The Opportunity:**
- One-time data capture with lifetime synthetic modeling potential
- Being part of the "special group" contributing to future research
- Self-service healthcare as empowerment, not abandonment

## Act III: Cultural Framework - The Mall (2-3 minutes)
**Beyond Retail Therapy:**
- The mall as second most important institution in Black households (after church, before school)
- Cultural significance for women now facing Graves disease
- Community gathering place, social hub, belonging
- Nostalgia for lost community spaces
- Alternative to Amazon - culturally rooted commerce

**The Vision:**
- Surrounding people with comfort while they wait
- Making healthcare navigation feel familiar and safe
- "Why settle for bearable when you can have comfortable?"

## Act IV: Technical Innovation - The AI Teaching Hospital (3-4 minutes)
**Introducing Kiro:**
- Chief resident leading specialist teams
- Real-time collaboration on complex cases
- Live demo of consilium in action

**The Demo Scenario:**
- Complex Graves disease case presented
- Kiro assembles team: Cultural specialist, Clinical evidence specialist, Community matching specialist
- Each agent contributes expertise
- Kiro synthesizes recommendations with acknowledgment
- Final culturally-appropriate, clinically-sound recommendation

**The Technical Secret:**
- Deterministic requirements methodology
- No decision-making conflicts between agents
- Pure collaborative execution
- Quality through multi-agent validation

## Act V: The Larger Vision (2-3 minutes)
**Research Infrastructure:**
- Building the largest dataset on Black women with Graves disease ever assembled
- Partnership with Amazon/TiDB for housing
- Modularized data collection (CLI approach)
- Transparent: "This is what we need, this is what we'll do with it, are you cool with that?"

**Community Impact:**
- Women become data scientists of their own health
- Action research approach - solving local problems
- First-class information and actionable data
- The right information at the right time with scaffolds to move forward

**The Legacy:**
- From personal story to systemic change
- From making mother comfortable to serving community
- From AI collaboration to research breakthrough
- "If you like it, then I love it" - scaled to serve thousands

## Closing Call to Action (30 seconds)
**The Choice:**
- Continue waiting for "them" to fix healthcare
- Or build self-service solutions that honor culture and empower community
- Join the special group creating the future of Black women's wellness research

---

**Total Runtime:** 10-12 minutes
**Tone:** Personal, authentic, technically sophisticated but accessible
**Key Message:** Personal motivation + Cultural understanding + Technical innovation = Systemic change

Ready for Collab to develop the full script?

Perfect. The outline is ready for Collab to develop into the full script.

The structure flows from deeply personal to technically sophisticated to systemically transformative. The progression from your birth story to building research infrastructure for thousands of Black women creates a compelling narrative arc.

A few technical notes for Collab when developing the full script:

**For the demo portion:** The Kiro consilium demonstration needs specific dialogue examples to show the multi-agent collaboration in action. Not just describing it, but showing actual agent interactions.

**For the data claims:** The "largest dataset ever" statement needs careful phrasing to be factually defensible. Emphasize the gap (2012 study, small samples) rather than making absolute claims about future outcomes.

**For the cultural mall section:** This is your strongest differentiator. Make sure the script captures the emotional resonance of mall-as-community-institution, not just retail space.

**For pacing:** The personal story opening needs time to breathe. Don't rush through the birth narrative - it's your emotional foundation.

Ready to hand this off to Collab for the full script development.

</details>
